# Tigers & Goats: Train and Evaluate
**Version 1.0** | **Last updated: 2026-10-17** | `TnG_Notebook.ipynb`

Standalone Python 3.10+ notebook for Colab or Jupyter, from the **Tigers & Goats (TNG_Falcon)** project.

Train either goat or tiger using Maskable PPO. Observations contain
23 board positions, goats eaten, and placement/movement phase; the action space has
115 actions. Legal-action masks constrain both training and evaluation.

This notebook adapts the environment/training workflow from TNG_Falcon, using
the original `x96_TnG.ipynb` as a reference. No clone, project
imports, pretrained downloads, interactive GUI, or replay files are required.

Run top to bottom. The default 2,048-step CPU run is a workflow check and does not result in a strong
policy. Longer experiments may use 10 million or more steps, but the required duration
depends on the opponent and settings; step count alone does not guarantee a strong policy. The evaluation section can be rerun without retraining. Human play is optional at the bottom.


### Notebook roadmap

1. **[Overview](#overview):** understand the board, actions, opponents, rewards, and workflow.
2. **[Configure your experiment](#configure-experiment):** edit the single settings cell, basic options first.
3. **[Dependency setup](#dependency-setup):** install packages into the active notebook kernel if needed.
4. **[Environment and reward reference](#environment-reference):** review the defaults and define the environment.
5. **[Training helpers](#training-helpers):** define validation, opponent sampling, logging, and saving.
6. **[Monitor training with TensorBoard](#monitor-tensorboard):** open the live dashboard before training.
7. **[Run training](#run-training):** execute the selected variations and curriculum phases.
8. **[Save outputs and continue later](#save-outputs):** keep results and resume checkpoints.
9. **[Evaluate models](#evaluate-models):** run the independent evaluation cells.
10. **[Play the game yourself](#human-play):** play goats or tigers on the clickable board.


<a id="overview"></a>

## Environment and training overview

This notebook uses one unified **Gymnasium environment, `TnGEnv`**, for both learner roles.
It includes goat placement and movement, tiger captures, legal-action masking,
configurable rewards, and Maskable PPO training. Select the learner and opponent
in the configuration cell below.

### Board Map

The board has **23 playable positions**, indexed `0–22` in the `board` array.
Three tigers start at positions `0`, `3`, and `4`. Goats place **15 pieces** before
entering the movement phase. Tigers win by capturing **6 goats**.

| | | 0 (b0) | | | |
| --- | --- | --- | --- | --- | --- |
| 1 (a1) | 2 (b1) | 3 (c1) | 4 (d1) | 5 (e1) | 6 (f1) |
| 7 (a2) | 8 (b2) | 9 (c2) | 10 (d2) | 11 (e2) | 12 (f2) |
| 13 (a3) | 14 (b3) | 15 (c3) | 16 (d3) | 17 (e3) | 18 (f3) |
| | 19 (b4) | 20 (c4) | 21 (d4) | 22 (e4) | |

`render()` prints an ASCII board: `.` = empty, `G` = goat, `T` = tiger.
The environment's `move_map` defines connections and directions; the table is a
position reference, not a full drawing of the legal edges.

### Observations and Actions

- **Observation:** a 25-element vector, `[board(23), goats_eaten, phase]`.
  Board values are `0=empty`, `1=goat`, `2=tiger`; phase is `0=placement`, `1=movement`.
- **Action space:** `Discrete(115)`, encoded as `pos * 5 + dir_code`.
  Decode with `pos = action // 5` and `dir_code = action % 5`.
- **Goat actions:** direction `0` places a goat during placement. Directions `1–4`
  identify movement through `move_map` during the movement phase.
- **Tiger actions:** select a tiger's position and legal direction. The environment
  resolves whether that direction is an ordinary move or a capture, including hub rules.
- **Masks:** `get_action_mask()` returns 115 Boolean values for the current learner.
  The notebook wraps this with `ActionMasker` for training and masked predictions
  during evaluation.

### Environment Functions

#### `TnGEnv(...)`

Important constructor arguments are `learner_role`, `tiger_ai`, `goat_opponent_ai`,
`reward_weights`, and optional `max_turns`. The constructor defaults to a goat learner
against Greedy Tigers; **this notebook's configuration also defaults to Greedy Tigers**.
The training helpers translate the configuration into these arguments.

`max_turns=None` uses `DEFAULT_KNOBS["MAX_TURNS"]`, currently **100**. Override it
through `REWARD_OVERRIDES` in the configuration cell. Model opponents use prediction
callbacks supplied by the helpers when a model file is configured.

#### `reset(seed=None, options=None)`

Resets the board, placement phase, counters, and move bookkeeping, and returns
`(observation, info)`. The notebook seeds Python, NumPy, and PyTorch as well as the
environment, because the scripted opponents also use global random generators.

#### `get_state()` and `render()`

`get_state()` returns the observation vector. `render()` prints the board and game
status; no interactive GUI is required.

#### `step(action)`

Returns `(observation, reward, terminated, truncated, info)`.

- **Goat learner:** the agent places or moves a goat, then the scripted tiger responds
  unless a terminal condition has already been reached.
- **Tiger learner:** the agent moves a tiger, then a Random or Model Goat responds
  unless a terminal condition has already been reached.
- Goats win when tigers have no legal moves. Tigers win at the capture threshold;
  the tiger-learner path also awards a tiger win when goats have no legal response.
- The turn limit ends an episode with `winner="MaxTimeout"`. The goat-learner path
  also detects repeated-state stalls, reported as `winner="RepeatTimeout"`.
- Invalid actions have separate soft/hard handling and can truncate an episode.
  Action masking is intended to prevent them. Inspect `info["reason"]` for details.

### Opponents and Mixing

| Learner | Configuration opponent | Behavior |
| --- | --- | --- |
| Goat | `"greedy"` | Capture-biased random Tiger: with default capture bias 1.0, takes a random available capture; otherwise selects a legal move |
| Goat | `"smart"` | Heuristic Tiger: prioritizes captures, anchor positions `0` and `9/10`, then stronger board squares |
| Tiger | `"random"` | Chooses a random legal Goat action |
| Tiger | `"model"` | Uses a saved Goat policy with deterministic masked predictions |

The Tiger policy methods are `_greedy_tiger_move(...)` and `_smart_tiger_move(...)`.
The notebook training configuration exposes the four matchups above; it does not
expose training goats against a saved Tiger model.

`MIX_PROB=None` uses a fixed opponent. Otherwise, at each reset, it selects Smart
instead of Greedy Tigers with that probability for a goat learner, or Model instead
of Random Goats for a tiger learner. A nonzero Model Goat probability requires a
valid model file. The helpers reject missing model files; within the environment,
a failed or illegal Model Goat prediction falls back to a random legal Goat action.

### Rewards

Although the environment's reward entry point is named `sparse_reward`, the current
implementation includes **role-specific shaping**, not just terminal rewards.

- **Goat learner:** step costs, tiger mobility, territory, tiger spread, center control,
  capture penalties, repetition penalties, and terminal outcomes. Some positive
  shaping decays during the movement phase.
- **Tiger learner:** step costs, capture rewards, mobility and center shaping, plus
  terminal win/loss and timeout terms.

Selected defaults (these are components, not necessarily the total step reward):

| Setting | Default effect |
| --- | --- |
| `REWARD_GOAT_WIN` | Goat win reward starts at `3.2`, reduced by `GOAT_WIN_TURN_DECAY * turns` (decay default `0.01`) |
| `REWARD_TIGER_WIN` | Goat learner's terminal loss component: `-3.2` |
| `REWARD_GOAT_EATEN` | Goat learner: `-0.35` per captured goat |
| `REWARD_TIGER_CAPTURE` | Tiger learner: `+0.35` per capture |
| `REWARD_TIGER_WIN_BONUS` | Tiger learner: `+3.2` terminal win bonus |
| `REWARD_TIGER_LOSS_PENALTY` | Tiger learner: subtract `3.2` on a goat win |
| `MAX_TIMEOUT_SCALE` | Turn-limit penalty scales the role's terminal loss term by `1.5` |
| `REPEAT_STALL_SCALE` | Goat learner's repeated-state timeout scales its loss term by `1.25` |

Use `REWARD_OVERRIDES` for custom rewards and game limits. The **Environment and
reward reference** section below lists every supported key, default, and meaning.

### Training and Evaluation Workflow

The default run trains a goat against Greedy Tigers for **2,048 timesteps** on CPU.
Training uses Maskable PPO, sequential vectorized environments, TensorBoard logging,
periodic checkpoints, and final saved models. Variations start independently;
curriculum phases continue the same model. Saved checkpoints can resume training.

The evaluation section runs separately, defaults to **100 episodes**, and can
evaluate newly trained or existing models. It reports wins, losses, timeouts, mean
reward, and mean episode length, and exports CSV/JSON results. A short training run
checks the workflow; use the configuration cell to increase training and evaluation
budgets before assessing policy strength.


<a id="configure-experiment"></a>

## Configure your experiment
Advanced training may require tweaking the environment and training cells. Edit **only the next code cell** for basic training. Start with Basic settings; optional and advanced
settings can stay unchanged. Edit the value to the right of `=`, keep text in quotes,
and use Python `None` (without quotes) to disable optional settings.

- Give each experiment a **unique name**. Goat learners face `"greedy"` or `"smart"`
  Tigers; tiger learners face `"random"` or `"model"` Goats.
- Model opponents need an uploaded/local goat `.zip`. Resuming needs a saved learner
  `.zip` with the same role and environment. Paths are relative to the notebook's
  working directory unless absolute. No files are downloaded automatically.
- After changing settings, **restart the kernel and Run All**. For evaluation only,
  set `EVAL_MODEL_PATH`, run configuration, setup, environment, and helper cells,
  **skip Run training**, then run the evaluation section.
- To rerun evaluation in the current kernel, edit evaluation settings in the configuration code cell,
  rerun that code cell and the evaluation cells. This does not update model weights.
- `cuda` requires a GPU runtime and CUDA-enabled PyTorch. CPU is a good default for
  this small MLP. Multiple environments here run sequentially, not as subprocesses.
- Advanced phase values override global defaults. Reward overrides merge in this
  order: environment defaults, global overrides, variation overrides, phase overrides.
  A curriculum continues one model; different variations start independent models.
- PPO completes whole rollouts, so actual timesteps may exceed a requested duration.

In [ ]:
# BASIC SETTINGS
EXPERIMENT_NAME = "First_Test"
LEARNER_ROLE = "goat"          # "goat" or "tiger"
OPPONENT = "greedy"             # Goat: "greedy", "smart"; Tiger: "random", "model"
TIMESTEPS = 1_000_000              # 2,048 steps checks the workflow; increase for substantive training

# COMMON OPTIONS
DEVICE = "cpu"                 # "cpu" or "cuda" (GPU runtime required)
NUM_ENVS = 8                   # Environments stepped sequentially; does not select CPU cores
SEED = 67                      # Integer from 0 through 2**32 - 1
OUTPUT_ROOT = "artifacts"      # Local folder or mounted persistent-storage path
CHECKPOINTS_PER_RUN = 10         # Per phase; 0 disables periodic checkpoints

# EVALUATION
EVAL_EPISODES = 100             # Per model/opponent; use more for reliable comparisons
EVAL_SEED = 1042               # Separate from training seed
# EVAL_OPPONENTS = None          # None: final training matchup, including its mix
EVAL_OPPONENTS = ["greedy", "smart"]  # Goat learner: compare both scripted Tigers
# EVAL_OPPONENTS = ["random", "model"]  # Tiger learner: model path required below
EVAL_DETERMINISTIC = True      # True: highest-probability legal action; False: sample
EVAL_MODEL_PATH = None         # None: newly trained models; or "path/to/learner.zip"
EVAL_SETTINGS_PATH = None      # Optional saved phase_*_settings.json for existing model

# OPTIONAL MODELS
RESUME_MODEL_PATH = None       # None: fresh learner; or "path/to/checkpoint.zip"
GOAT_MODEL_PATH = None         # Required for Model Goat or any nonzero goat-model mix
# Resume keeps the saved architecture and PPO settings. TIMESTEPS adds more training.
# Use the original learner role/reward/game settings, or intentionally change them.

# ADVANCED TRAINING
N_STEPS = 128                  # Rollout steps per environment (positive integer)
BATCH_SIZE = 64                # >1; must divide N_STEPS * NUM_ENVS for fresh models
N_EPOCHS = 5                   # PPO optimization epochs per rollout
LEARNING_RATE = 3e-4           # Positive learning rate
GAMMA = 0.99                   # Discount factor, 0 through 1
ENT_COEF = 0.02                # Nonnegative entropy coefficient
NET_ARCH = [256, 256, 256]     # Positive hidden-layer widths used by the training helpers cell

# CUSTOM EXPERIMENTS
MIX_PROB = None                # None: fixed opponent; otherwise probability 0..1
# Goat learner: P(Smart Tiger), otherwise Greedy Tiger.
# Tiger learner: P(Model Goat), otherwise Random Goat. Set GOAT_MODEL_PATH if >0.
REWARD_OVERRIDES = {}          # Keys from DEFAULT_KNOBS in the environment definition cell below
# REWARD_OVERRIDES = {"REWARD_STEP": -0.002, "MAX_TURNS": 100}

# VARIATIONS is a dictionary: {experiment_name: training_settings}.
# Each entry trains a separate model. The key (e.g. "baseline") names its results.
# The value tells that model how to train:
#   None       = use the settings above.
#   {...}      = change settings for one training stage (a "phase").
#   [{...}, ...] = train the same model through several phases, in order.
# A new phase keeps what the model learned; a new variation starts separately.
# These training phases are different from the game's placement/movement phases.
VARIATIONS = None             # One baseline experiment using the settings above

# Optional settings inside a phase dictionary:
#   timesteps: how many training steps to run in this phase (across all environments).
#   opponent: goat learner: "greedy" or "smart"; tiger learner: "random" or "model".
#   mix_prob: None = fixed opponent; 0.5 = 50% Smart Tiger / 50% Greedy Tiger
#       for goats, or 50% Model Goat / 50% Random Goat for tigers, chosen each game.
#   checkpoints_per_run: how many checkpoints to aim to save; 0 = no periodic saves.
#   goat_model_path: saved goat model file, needed when tigers face Model Goat.
#   reward_overrides: reward/game settings to change, e.g. {"REWARD_STEP": -0.002}.
# Leave out a setting to use its value from above, not from the previous phase.

# Example for a goat learner: uncomment to replace VARIATIONS = None.
# VARIATIONS = {
#     "baseline": None,                  # Model 1: use the settings above
#     "short_run": {"timesteps": 2_048}, # Model 2: change only the step count
#     "curriculum": [                   # Model 3: two phases, same model
#         {"opponent": "greedy", "mix_prob": None, "timesteps": 2_048},
#         {"opponent": "smart", "mix_prob": None, "timesteps": 2_048},
#     ],                                # About 4,096 steps in total
# }

# To share settings across phases, put them beside a "phases" key.
# Its value is a list of phase dictionaries. A phase can change any shared setting.
# Reward dictionaries combine; a phase's value wins when the same key appears twice.
# VARIATIONS = {"curriculum": {
#     "timesteps": 2_048, "mix_prob": None,  # Apply to both phases
#     "phases": [{"opponent": "greedy"}, {"opponent": "smart"}],
# }}
# PPO and device settings above apply to every experiment.

<a id="dependency-setup"></a>

## Dependency setup
Run the next code cell in the notebook kernel. If installation asks for a restart, restart
and run again from the configuration cell. Installation needs internet on first use.

This cell also imports the libraries used throughout the notebook and defines
`in_colab` once for TensorBoard and human play. Run it after every kernel restart,
even when the packages are already installed.


In [2]:
%pip install "stable-baselines3[extra]>=2.7,<3" "sb3-contrib>=2.7,<3" "gymnasium>=1.2,<2" torch numpy pandas tensorboard

import sys
from IPython import get_ipython

# Shared notebook environment check; reused by TensorBoard and human play.
shell = get_ipython()
in_colab = (
    "google.colab" in sys.modules
    or type(shell).__module__.startswith("google.colab")
)

# Keep the local widget version compatible with VS Code.
if not in_colab:
    %pip install "ipywidgets==8.1.8"

# Shared imports for the environment, training, evaluation, and human play.
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random
import copy
import csv
import json
import math
import re
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from collections import deque, Counter
import pandas as pd
import torch
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.callbacks import BaseCallback, CallbackList, CheckpointCallback
from stable_baselines3.common.logger import configure
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from urllib.request import urlopen
from IPython.display import IFrame, display
from tensorboard import program
import shlex
import ipywidgets as widgets
from html import escape

if in_colab:
    from google.colab import output


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


<a id="environment-reference"></a>

## Environment and reward reference
The next code cell defines `TnGEnv`, the game environment used by the training
and evaluation cells. It includes both learner roles, board rules, action masks,
scripted opponents, rewards, and episode endings; running it does not start training.
Use the configuration cell's `REWARD_OVERRIDES` to change these defaults.

### Reward and game settings

| Key | Default | Meaning |
| --- | --- | --- |
| `REWARD_GOAT_WIN` | `3.2` | reward when goats immobilize all tigers |
| `REWARD_TIGER_WIN` | `-3.2` | penalty when tigers win by eating goats |
| `REWARD_TIGER_CAPTURE` | `0.35` | reward per goat captured (tiger learner) |
| `REWARD_TIGER_WIN_BONUS` | `3.2` | reward for tiger win (tiger learner) |
| `REWARD_TIGER_LOSS_PENALTY` | `3.2` | penalty when tiger loses or times out (tiger learner) |
| `REWARD_STEP` | `-0.001` | per-step penalty to discourage stalling |
| `REWARD_GOAT_EATEN` | `-0.35` | penalty for each goat captured by tigers |
| `MAX_TIMEOUT_SCALE` | `1.5` | scale for turn-limit (max turns) timeout penalty |
| `REPEAT_STALL_SCALE` | `1.25` | scale for stall-timeout (repeat-state) penalty |
| `MAX_TURNS` | `100` | max turns before enforcing max-timeout |
| `MOVE_STEP_BASE` | `-0.01` | base negative reward for moving-phase steps |
| `MOVE_STEP_SLOPE` | `-0.01` | step penalty increases as move_steps grows |
| `MOVE_STEP_MIN` | `-0.6` | minimum move penalty (lower bound) |
| `GOAT_WIN_TURN_DECAY` | `0.01` | goat win reward reduced slightly per turn |
| `REWARD_BLOCK_TIGER` | `0.08` | reward for reducing tiger mobility |
| `REWARD_BUBBLE_SPACE` | `0.02` | reward for increasing goat "bubble" territory |
| `REWARD_CLUSTER_TIGERS` | `0.02` | reward for spreading tigers apart |
| `REWARD_CENTER_GOAT` | `0.03` | reward for goats occupying strong center nodes |
| `REWARD_CENTER_TIGER` | `-0.03` | penalty for tigers holding center nodes |
| `NEAR_LOCK_BONUS` | `0.05` | bonus when goats push tigers into near-lock states |
| `LATE_GAME_START_TURN` | `40` | turn at which late-game shaping begins to scale |
| `MOBILITY_BACKSLIDE_SCALE` | `0.5` | penalty multiplier when goats worsen tiger mobility |
| `GOAT_SHAPING_DECAY_BASE` | `0.95` | exponential decay for positive goat shaping in moving phase |
| `REWARD_INVALID_SOFT` | `-0.05` | small penalty for soft invalid actions |
| `REWARD_INVALID_HARD` | `-1.0` | large penalty for hard invalid actions (terminate) |
| `REWARD_REPEAT_STATE` | `-0.5` | (Quadratic along count) penalty for repeating a previous board state |
| `MAX_REPEATS` | `4` | how many repeats before a stall timeout triggers |
| `BASE_TIGER_CAPTURE_BIAS` | `1.0` | how greedy tigers are toward captures (1.0 = full greedy) |

### Environment interface and extension points

The environment provides one `Discrete(115)` interface for both learner roles,
legal-action masks, placement/movement rules, repetition handling, and time limits.
`reset()` returns `(observation, info)`; `step()` returns
`(observation, reward, terminated, truncated, info)`. Game wins are terminal outcomes;
limits and invalid-action endings use truncation as described in the overview.
The `info` dictionary carries outcome/reason, role, move, and reward-component data
where supplied by the relevant transition and reward function.

For advanced environment customization, the constructor accepts `reward_fn` and
stores it as `self.reward_fn`. The default is the role-aware `sparse_reward` method,
which includes the shaping described above. A custom reward callable receives
`(previous_observation, action, observation, terminated, truncated, info)` and returns
one scalar reward. This requires editing environment construction in the helpers;
ordinary experiments should use `REWARD_OVERRIDES` in the configuration cell.

The underlying environment also supports `tiger_ai="model"` with a
`tiger_model_predict_fn` callback. The notebook training controls expose Greedy/Smart
Tigers and Random/Model Goats; saved Tiger opponents require a helper extension.

### Environment construction examples

These examples explain the API; they are not extra configuration cells. They become
available after executing the environment definition below:

```python
env = TnGEnv()  # Goat learner against Greedy Tigers
env = TnGEnv(tiger_ai=TIGER_AI_SMART)  # Goat learner against Smart Tigers
env = TnGEnv(learner_role=TIGER_LEARNER)  # Tiger learner against Random Goats
env = TnGEnv(max_turns=150)  # Explicit turn limit
env = TnGEnv(reward_weights={"REWARD_BUBBLE_SPACE": 0.0})  # Disable this shaping term
```

Choose one construction, call `env.reset()`, use the legal-action mask for steps,
and call `env.close()` when finished. The notebook's training/evaluation helpers
perform that lifecycle for normal experiments.

### How the notebook cells fit together

The configuration code cell supplies experiment settings. The environment
definition cell supplies `TnGEnv` and its default tuning values. The training
helpers cell creates environments, handles opponent mixing, logs metrics, and
saves results when the Run training code cell is executed. The evaluation cells
compare saved or newly trained models using the evaluation settings in the
configuration code cell. The optional human-play cells at the bottom provide a clickable board.


In [3]:
#  GLOBAL CONSTANTS - Game Rules

BOARD_SIZE                = 23          # board positions (0-22)
DIR_CODES                 = 5           # direction codes (0-4)
TOTAL_GOATS_TO_PLACE      = 15          # goats placed before moving phase begins
GOATS_EATEN_FOR_TIGER_WIN = 6           # tiger wins after this many goats eaten
TIGER_START_POSITIONS     = [0, 3, 4]   # starting indices for tigers
KEY_CENTERS               = [0, 9, 10, 15, 16] # top hub + center spots to control

# Debug flag to avoid slow prints in training
DEBUG_INVALID = False

TIGER_AI_GREEDY = "greedy"
TIGER_AI_SMART = "smart"
TIGER_AI_MODEL = "model"
VALID_TIGER_AI = {TIGER_AI_GREEDY, TIGER_AI_SMART, TIGER_AI_MODEL}
GOAT_LEARNER = "goat"
TIGER_LEARNER = "tiger"
GOAT_AI_RANDOM = "random"
GOAT_AI_MODEL = "model"


#  GLOBAL TUNING KNOBS
#  Configurable reward magnitudes, shaping scales, and episode limits.
#  Override these values with REWARD_OVERRIDES in the configuration code cell.

DEFAULT_KNOBS = {

    # Goat-learner terminal rewards
    "REWARD_GOAT_WIN":          3.2,    # reward when goats immobilize all tigers
    "REWARD_TIGER_WIN":        -3.2,    # penalty when tigers win by eating goats

    # Tiger-learner rewards (positive magnitudes)
    "REWARD_TIGER_CAPTURE":     0.35,   # reward per goat captured (tiger learner)
    "REWARD_TIGER_WIN_BONUS":   3.2,    # reward for tiger win (tiger learner)
    "REWARD_TIGER_LOSS_PENALTY": 3.2,   # penalty when tiger loses or times out (tiger learner)

    # Goat-learner step & capture shaping
    "REWARD_STEP":             -0.001,  # per-step penalty to discourage stalling
    "REWARD_GOAT_EATEN":       -0.35,   # penalty for each goat captured by tigers

    # Timeout scales
    "MAX_TIMEOUT_SCALE":        1.5,    # scale for turn-limit (max turns) timeout penalty
    "REPEAT_STALL_SCALE":       1.25,   # scale for stall-timeout (repeat-state) penalty
    "MAX_TURNS":                100,    # max turns before enforcing max-timeout

    # Move penalty shaping
    "MOVE_STEP_BASE":          -0.01,   # base negative reward for moving-phase steps
    "MOVE_STEP_SLOPE":         -0.01,   # step penalty increases as move_steps grows
    "MOVE_STEP_MIN":           -0.6,    # minimum move penalty (lower bound)

    # Decay for goat win  
    "GOAT_WIN_TURN_DECAY":      0.01,   # goat win reward reduced slightly per turn

    # Shaping magnitudes  
    "REWARD_BLOCK_TIGER":       0.08,   # reward for reducing tiger mobility
    "REWARD_BUBBLE_SPACE":      0.02,   # reward for increasing goat "bubble" territory
    "REWARD_CLUSTER_TIGERS":    0.02,   # reward for spreading tigers apart
    "REWARD_CENTER_GOAT":       0.03,   # reward for goats occupying strong center nodes
    "REWARD_CENTER_TIGER":     -0.03,   # penalty for tigers holding center nodes

    # Special shaping
    "NEAR_LOCK_BONUS":          0.05,   # bonus when goats push tigers into near-lock states
    "LATE_GAME_START_TURN":     40,     # turn at which late-game shaping begins to scale
    "MOBILITY_BACKSLIDE_SCALE": 0.5,    # penalty multiplier when goats worsen tiger mobility
    "GOAT_SHAPING_DECAY_BASE":  0.95,   # exponential decay for positive goat shaping in moving phase

    # Anti-repeat penalty
    "REWARD_INVALID_SOFT":     -0.05,   # small penalty for soft invalid actions
    "REWARD_INVALID_HARD":     -1.0,    # large penalty for hard invalid actions (terminate)
    "REWARD_REPEAT_STATE":     -0.5,    # (Quadratic along count) penalty for repeating a previous board state
    "MAX_REPEATS":              4,      # how many repeats before a stall timeout triggers

    # Tiger AI difficulty
    "BASE_TIGER_CAPTURE_BIAS": 1.0,     # how greedy tigers are toward captures (1.0 = full greedy)
}


#  BOARD COORDINATES - Human-Readable Labels

COORD_LABELS = [
    "b0",                                    # 0
    "a1", "b1", "c1", "d1", "e1", "f1",      # 1-6
    "a2", "b2", "c2", "d2", "e2", "f2",      # 7-12
    "a3", "b3", "c3", "d3", "e3", "f3",      # 13-18
    "b4", "c4", "d4", "e4",                  # 19-22
]

# Special hub capture jumps from node 0 (b0)
HUB0_CAPTURE_JUMP = {
    1: 8,   # 0 -> 2 (b1) -> 8 (b2)
    2: 9,   # 0 -> 3 (c1) -> 9 (c2)
    3: 10,  # 0 -> 4 (d1) -> 10 (d2)
    4: 11,  # 0 -> 5 (e1) -> 11 (e2)
}


#  Board index to coordinate helper

def idx_to_coord(idx: int) -> str:
    """Safely convert a board index to its coordinate label."""
    if 0 <= idx < len(COORD_LABELS):
        return COORD_LABELS[idx]
    return str(idx)


#  BASE ENVIRONMENT

class TnGEnv(gym.Env):
    """
    Tigers and Goats Environment.

    Observation (Box(25,)):
      - 23 board cells: 0=empty, 1=goat, 2=tiger
      - 1 value: goats eaten (0..6)
      - 1 value: phase (0=placing, 1=moving)

    Action (Discrete(115)):
      - flat action index in [0, 114]
      - decode rule: pos = action // DIR_CODES, dir_code = action % DIR_CODES
    """

    #  Group: Core Setup and Configuration
    #  Initializes persistent env state and shared tuning knobs.
    #  These members are prerequisites for all runtime behavior.
    def __init__(
            self, 
            reward_weights=None, 
            tiger_ai: str = TIGER_AI_GREEDY, 
            learner_role: str = GOAT_LEARNER, 
            goat_opponent_ai: str = GOAT_AI_RANDOM, 
            goat_model_predict_fn=None, 
            tiger_model_predict_fn=None,
            reward_fn=None,
            max_turns: int | None = None,
            ):
        """Initialize environment state, tuning knobs, spaces, and board topology."""
        super(TnGEnv, self).__init__()

        # Numeric tuning knobs (constants + shaping magnitudes + scales)
        self.knobs = dict(DEFAULT_KNOBS)

        self.learner_role = (learner_role or GOAT_LEARNER).lower()
        self.goat_opponent_ai = (goat_opponent_ai or GOAT_AI_RANDOM).lower()
        self._goat_model_predict_fn = goat_model_predict_fn  # callable(obs, mask)->flat action
        self._tiger_model_predict_fn = tiger_model_predict_fn  # callable(obs, mask)->flat action
        self.reward_fn = reward_fn or self.sparse_reward

        # Load any tuning values passed to this function.
        if reward_weights is not None:
            for key, val in reward_weights.items():
                if key in self.knobs:
                    self.knobs[key] = val
                else:
                    print(f"[warning] Unknown tuning key: {key}")

        configured_max_turns = self.knobs["MAX_TURNS"] if max_turns is None else max_turns
        self.max_turns = int(configured_max_turns)
        if self.max_turns <= 0:
            raise ValueError(f"max_turns must be > 0, got: {configured_max_turns}")
        # Keep _w("MAX_TURNS") and explicit max_turns aligned.
        self.knobs["MAX_TURNS"] = self.max_turns

        self.tiger_ai = (tiger_ai or TIGER_AI_GREEDY).lower()
        if self.tiger_ai not in VALID_TIGER_AI:
            raise ValueError(f"tiger_ai must be one of {VALID_TIGER_AI}")
        if self.learner_role not in {GOAT_LEARNER, TIGER_LEARNER}:
            raise ValueError(f"learner_role must be one of: {GOAT_LEARNER}, {TIGER_LEARNER}")
        if self.goat_opponent_ai not in {GOAT_AI_RANDOM, GOAT_AI_MODEL}:
            raise ValueError(f"goat_opponent_ai must be one of: {GOAT_AI_RANDOM}, {GOAT_AI_MODEL}")

        # Observation space
        self.observation_space = spaces.Box(
            low=np.array([0] * BOARD_SIZE + [0] + [0], dtype=np.int8),
            high=np.array([2] * BOARD_SIZE + [6] + [1], dtype=np.int8),
            shape=(BOARD_SIZE + 2,),
            dtype=np.int8,
        )

        # Action space (flat)
        self.action_space = spaces.Discrete(BOARD_SIZE * DIR_CODES)

        # Game state variables
        self.terminate = False
        self.board = np.zeros(BOARD_SIZE, dtype=np.int8)
        for idx in TIGER_START_POSITIONS:
            self.board[idx] = 2

        self.eaten = 0          # goats eaten by tigers
        self.phase = 0          # 0 = placing phase, 1 = moving phase
        self.goats_placed = 0

        # For reward shaping
        self.prev_tiger_moves = 0
        self.prev_unreachable = 0
        self.prev_tiger_spread = 0.0

        # Repetition tracker
        self.turns = 0
        self.move_steps = 0     # Counts moves taken in move phase
        self.state_history = {}

        # For human-readable move tracking in render()
        self.last_move_desc = ""

        # Move map:
        # pos -> {dir_code: destination_pos}
        self.move_map = {
            0: {1: 2, 2: 3, 3: 4, 4: 5},  # b0 special
            1: {2: 2, 3: 7},                 # a1
            2: {1: 0, 2: 3, 3: 8, 4: 1},     # b1
            3: {1: 0, 2: 4, 3: 9, 4: 2},     # c1
            4: {1: 0, 2: 5, 3: 10, 4: 3},    # d1
            5: {1: 0, 2: 6, 3: 11, 4: 4},    # e1
            6: {3: 12, 4: 5},                # f1

            7: {1: 1, 2: 8, 3: 13},          # a2
            8: {1: 2, 2: 9, 3: 14, 4: 7},    # b2
            9: {1: 3, 2: 10, 3: 15, 4: 8},   # c2
            10: {1: 4, 2: 11, 3: 16, 4: 9},  # d2
            11: {1: 5, 2: 12, 3: 17, 4: 10}, # e2
            12: {1: 6, 3: 18, 4: 11},        # f2

            13: {1: 7, 2: 14},               # a3
            14: {1: 8, 2: 15, 3: 19, 4: 13}, # b3
            15: {1: 9, 2: 16, 3: 20, 4: 14}, # c3
            16: {1: 10, 2: 17, 3: 21, 4: 15},# d3
            17: {1: 11, 2: 18, 3: 22, 4: 16},# e3
            18: {1: 12, 4: 17},              # f3

            19: {1: 14, 2: 20},              # b4
            20: {1: 15, 2: 21, 4: 19},       # c4
            21: {1: 16, 2: 22, 4: 20},       # d4
            22: {1: 17, 4: 21},              # e4
        }

        # Precompute all-pairs shortest path lengths on the static board graph
        self._dist_matrix = self._precompute_all_pairs_dist()

        # Pre-allocated visited buffer for bubble BFS
        self._visited_buffer = np.zeros(BOARD_SIZE, dtype=bool)

        # Initial valid moves
        self.valid_moves = []
        self._update_valid_moves()
    # end def __init__()

    def _w(self, key) -> float:
        """
        Knob accessor:
            - If key is a known tuning knob -> return value
            - If unknown -> throw an error
        """
        if key in self.knobs:
            return self.knobs[key]
        
        raise KeyError(
            f"[TnGEnv] Unknown tuning key '{key}'. "
            f"Valid knob keys: {list(self.knobs.keys())}."
        )


    #  Group: Public Gym API
    #  Entry points expected by Gymnasium and training loops.
    #  Keeps reset/step/state/render/mask behavior in one place.
    def reset(self, seed=None, options=None):
        """Reset board, game state, and valid moves."""
        super().reset(seed=seed)

        # Normalize + validate tiger_ai (important if wrappers modify it between episodes)
        self.tiger_ai = (self.tiger_ai or TIGER_AI_GREEDY).lower()
        if self.tiger_ai not in VALID_TIGER_AI:
            raise ValueError(f"tiger_ai must be one of {VALID_TIGER_AI}")

        # Snapshot tiger used for THIS episode (critical for mixed training)
        self._episode_tiger_ai = self.tiger_ai

        self.board[:] = 0
        for idx in TIGER_START_POSITIONS:
            self.board[idx] = 2

        self.eaten = 0
        self.phase = 0
        self.goats_placed = 0
        self.terminate = False

        self.turns = 0
        self.move_steps = 0
        self.state_history = {}

        # clear last move description
        self.last_move_desc = ""

        # initial tiger mobility + geometry metrics
        tiger_opts = self._tiger_moves()
        self.prev_tiger_moves = len(tiger_opts)
        self.prev_unreachable = self._compute_unreachable_safe_cells()
        self.prev_tiger_spread = self._tiger_spread()

        self._update_valid_moves()

        obs = self.get_state()
        info = {
            "action_mask": self.get_action_mask(),
            "tiger_ai": self.tiger_ai,
            "tiger_ai_id": float(1 if self.tiger_ai == TIGER_AI_SMART else 0),
            "episode_tiger_ai": self._episode_tiger_ai,
            "episode_tiger_ai_id": float(1 if self._episode_tiger_ai == TIGER_AI_SMART else 0)
            }
        return obs, info

    def step(self, action):
        """Gym step: execute learner-role transition, then shape reward from step info."""
        prev_obs = self.get_state().copy()
        if self.learner_role == TIGER_LEARNER:
            result = self._step_tiger_transition(action)
        else:
            result = self._step_goat_transition(action)
        return self._apply_reward_fn(prev_obs, action, result)

    def get_state(self):
        """
        Flattened state: [board(23), eaten(1), phase(1)] as int8.
        """
        return np.concatenate(
            (self.board, [self.eaten, self.phase])
        ).astype(np.int8)

    def render(self, show_coords=True):
        """
        Polished ASCII render for easier human analysis.
            - Empty cells shown as '0' instead of '.'
            - Cleaner alignment and spacing
        - Optional coordinate labels
        - Status bar showing game metrics
        """

        # Updated symbol map: ie. empty = 0,...
        symbols = {0: "0", 1: "G", 2: "T"}    
        cells = [symbols[v] for v in self.board]
        goats_eaten = self.eaten
        phase_str = "Place" if self.phase == 0 else "Move"
    
        # pull board coordinates from global COORD_LABELS
        coord = (
            [f" {COORD_LABELS[i]} " for i in range(BOARD_SIZE)] 
            if show_coords else [""] * BOARD_SIZE
        )
    
        # Build each visual line
        line0 = f"                  {cells[0]}{coord[0]}"
        line1 = (
            f"   {cells[1]}{coord[1]} {cells[2]}{coord[2]} "
            f"{cells[3]}{coord[3]} {cells[4]}{coord[4]} "
            f"{cells[5]}{coord[5]} {cells[6]}{coord[6]}"
        )
        line2 = (
            f"   {cells[7]}{coord[7]} {cells[8]}{coord[8]} "
            f"{cells[9]}{coord[9]} {cells[10]}{coord[10]} "
            f"{cells[11]}{coord[11]} {cells[12]}{coord[12]}"
        )
        line3 = (
            f"   {cells[13]}{coord[13]} {cells[14]}{coord[14]} "
            f"{cells[15]}{coord[15]} {cells[16]}{coord[16]} "
            f"{cells[17]}{coord[17]} {cells[18]}{coord[18]}"
        )
        line4 = (
            f"         {cells[19]}{coord[19]} {cells[20]}{coord[20]} "
            f"{cells[21]}{coord[21]} {cells[22]}{coord[22]}"
        )
    
        # Status panel
        status = (
            f"-----------------------------------------\n"
            f" Phase: {phase_str:<5} | Goats Eaten: {goats_eaten} | Turn: {self.turns}\n"
            f"-----------------------------------------"
        )
    
        # Last-move description
        if getattr(self, "last_move_desc",""):
            status += f"\n Last move: {self.last_move_desc}"

        print(
            "\n"
            "============== TIGERS & GOATS BOARD ==============\n"
            f"{line0}\n{line1}\n{line2}\n{line3}\n{line4}\n{status}\n"
        )
    # end def render()

    def get_action_mask(self):
        """
        Returns a Boolean mask of shape (BOARD_SIZE * DIR_CODES,),
        where each index corresponds to (pos, dir_code) = (i // DIR_CODES, i % DIR_CODES).

        - Goat learner: uses self.valid_moves in transition-safe form:
            - pair: [pos, dir_code] or (pos, dir_code)
            - flat: int action id
        - Tiger learner: uses _tiger_moves(include_dir=True) to mark legal tiger actions.
        """
        mask = np.zeros(BOARD_SIZE * DIR_CODES, dtype=bool)

        if getattr(self, "learner_role", GOAT_LEARNER) == TIGER_LEARNER:
            for from_pos, _to_pos, _cap, dir_code in self._tiger_moves(include_dir=True):
                flat = from_pos * DIR_CODES + dir_code
                if 0 <= flat < mask.size:
                    mask[flat] = True
            return mask

        for mv in self.valid_moves:
            if np.isscalar(mv):
                flat = int(mv)
            else:
                pos, dir_code = mv
                flat = self.encode_action(pos, dir_code)
            if 0 <= flat < mask.size:
                mask[flat] = True

        return mask


    #  Group: Action Encoding and Validation
    #  Converts between flat actions and structured moves.
    #  Centralizes defensive parsing/normalization for transitions.
    def encode_action(self, pos: int, dir_code: int) -> int:
        """Map (board position, direction code) -> flat Discrete action id."""
        return int(pos) * DIR_CODES + int(dir_code)

    def decode_action(self, action):
        """Decode a flat Discrete(115) action id into `(pos, dir_code)`."""
        if isinstance(action, np.ndarray):
            arr = np.asarray(action)
            if arr.size != 1:
                raise ValueError(f"Unrecognized action format: {action} (type {type(action)})")
            action = arr.item()

        if not np.isscalar(action):
            raise ValueError(f"Unrecognized action format: {action} (type {type(action)})")

        a = int(action)
        return a // DIR_CODES, a % DIR_CODES

    def _action_to_flat(self, action):
        """Best-effort conversion from a scalar action into a flat action id."""
        if isinstance(action, np.ndarray):
            arr = np.asarray(action)
            if arr.size != 1:
                return None
            action = arr.item()
        if np.isscalar(action):
            return int(action)
        return None

    def _safe_decode_action(self, action):
        """Decode action into (pos, dir) and return None on malformed input."""
        try:
            pos, dir_code = self.decode_action(action)
            return int(pos), int(dir_code)
        except Exception:
            return None

    def _build_step_info(self, **extra):
        """Build a consistent info payload and apply call-site overrides."""
        info = {
            "reason": None,
            "winner": None,
            "goat_action": None,
            "goat_decoded": None,
            "tiger_move": None,
            "phase": int(self.phase),
            "goats_eaten": int(self.eaten),
            "goats_placed": int(self.goats_placed),
            "turn_counter": int(self.turns),
            "learner_role": self.learner_role,
            "action_mask": self.get_action_mask(),
            "tiger_ai": self.tiger_ai,
            "tiger_ai_id": float(1 if self.tiger_ai == TIGER_AI_SMART else 0),
            "episode_tiger_ai": self._episode_tiger_ai,
            "episode_tiger_ai_id": float(1 if self._episode_tiger_ai == TIGER_AI_SMART else 0),
            "goat_opponent_ai": self.goat_opponent_ai,
            "reward_step_component": 0.0,
            "reward_near_lock_component": 0.0,
            "reward_block_component": 0.0,
            "reward_bubble_component": 0.0,
            "reward_cluster_component": 0.0,
            "reward_center_component": 0.0,
            "reward_goat_eaten_component": 0.0,
            "reward_repeat_component": 0.0,
            "reward_terminal_component": 0.0,
            "reward_decay_mult": 1.0,
            "reward_total": 0.0,
        }
        info.update(extra)
        return info

    def _set_reward_components(self, info, **components):
        """Attach scalar reward breakdown fields to info for logging/debugging."""
        for key, value in components.items():
            info[key] = float(value)


    #  Group: Step Info and Reward Pipeline
    #  Builds standardized info payloads and computes shaped rewards.
    #  Keeps reward flow consistent across goat/tiger learner modes.
    def _apply_reward_fn(self, prev_obs, action, result):
        """Run the configured reward function for the transition result."""
        obs, reward, terminated, truncated, info = result
        final_reward = self.reward_fn(
            prev_obs,
            action,
            obs,
            bool(terminated),
            bool(truncated),
            info,
        )
        return obs, float(final_reward), bool(terminated), bool(truncated), info

    def sparse_reward(self, prev_obs, action, obs, terminated, truncated, info):
        """
        Role-aware sparse reward entry point.

        Purpose:
        - The environment supports two training modes (goat learner, tiger learner).
        - Both modes share transition machinery but use different reward objectives.
        - This function routes to the correct reward builder using the role recorded
          in `info` (preferred) with `self.learner_role` as fallback.
        """
        role = info.get("learner_role", self.learner_role)
        if role == TIGER_LEARNER:
            return self._sparse_reward_tiger(prev_obs, action, obs, terminated, truncated, info)
        return self._sparse_reward_goat(prev_obs, action, obs, terminated, truncated, info)

    def _sparse_reward_goat(self, prev_obs, action, obs, terminated, truncated, info):
        """
        Goat-learner reward from transition summary in `info`.

        High-level composition:
        1) Early-return terminal/invalid cases from `reason`
        2) Per-step shaping and tactical shaping
        3) Material and position shaping
        4) Outcome penalties (tiger win / timeout) or repeat-state penalty

        Important `info` fields consumed here:
        - reason, step_penalty, near_lock, late
        - delta_moves, delta_bubble, delta_spread, center_score
        - goats_eaten_this_turn, repeat_prev_count
        """
        reason = info.get("reason")

        # 1) Immediate terminal/invalid outcomes
        # These bypass incremental shaping to keep terminal reward semantics explicit.
        if reason == "already_terminated":
            self._set_reward_components(info, reward_total=0.0)
            return 0.0
        if reason == "invalid_soft":
            reward_total = self._w("REWARD_INVALID_SOFT")
            self._set_reward_components(info, reward_total=reward_total)
            return reward_total
        if reason == "invalid_hard":
            reward_total = self._w("REWARD_INVALID_HARD")
            self._set_reward_components(info, reward_total=reward_total)
            return reward_total
        if reason == "goat_win_no_tiger_moves":
            # Goat terminal win bonus, with optional late-turn decay.
            reward_total = (
                self._w("REWARD_GOAT_WIN")
                - self._w("GOAT_WIN_TURN_DECAY") * self.turns
            )
            self._set_reward_components(
                info,
                reward_terminal_component=reward_total,
                reward_total=reward_total,
            )
            return reward_total

        # 2) Base step + tactical shaping
        # Step penalty comes from step_transition code (placing/moving can differ).
        step_component = float(info.get("step_penalty", self._w("REWARD_STEP")))
        near_lock_component = 0.0
        block_component = 0.0
        bubble_component = 0.0
        cluster_component = 0.0
        center_component = 0.0
        goat_eaten_component = 0.0
        repeat_component = 0.0
        terminal_component = 0.0

        phase = int(info.get("phase", self.phase))
        move_steps = int(info.get("move_steps", self.move_steps))

        # Exponential decay based on number of steps
        decay_mult = (
            self._w("GOAT_SHAPING_DECAY_BASE") ** move_steps
            if phase == 1 else 1.0
        )

        # Get delta and goat eaten stats which are computed in the goat_step_transition function
        
        # Mobility delta is defined from goat perspective:
        #   delta_moves > 0 means goats reduced tiger options (good for goats).
        #   delta_moves < 0 means goats increased tiger options (bad backslide).
        delta_moves = int(info.get("delta_moves", 0))
        delta_bubble = int(info.get("delta_bubble", 0))
        delta_spread = float(info.get("delta_spread", 0.0))
        goats_eaten_this_turn = int(info.get("goats_eaten_this_turn", 0))

        terminal_loss_reason = reason in {
            "tiger_win_capture_threshold",
            "max_timeout",
            "repeat_timeout",
        }

        if not terminal_loss_reason:
            # Near lock is computed in the goat step transition function
            if bool(info.get("near_lock", False)):
                near_lock_component = self._w("NEAR_LOCK_BONUS") * decay_mult

            if delta_moves > 0:
                block_component = (
                    abs(self._w("REWARD_BLOCK_TIGER"))
                    * float(np.tanh(delta_moves))
                    * decay_mult
                )
            elif delta_moves < 0:
                block_component = (
                    abs(self._w("REWARD_BLOCK_TIGER"))
                    * self._w("MOBILITY_BACKSLIDE_SCALE")
                    * delta_moves
                )

            if delta_bubble > 0:
                bubble_component = (
                    self._w("REWARD_BUBBLE_SPACE")
                    * float(np.tanh(delta_bubble))
                    * decay_mult
                )

            if delta_spread > 0:
                cluster_component = (
                    self._w("REWARD_CLUSTER_TIGERS")
                    * float(np.tanh(delta_spread))
                    * decay_mult
                )

            # Goat center shaping is intentionally limited to the placing phase.
            if phase == 0:
                center_component = float(info.get("center_score", 0.0))

        if goats_eaten_this_turn > 0:
            goat_eaten_component = (
                self._w("REWARD_GOAT_EATEN")
                * goats_eaten_this_turn
            )

        if reason == "tiger_win_capture_threshold":
            terminal_component = self._w("REWARD_TIGER_WIN")
        elif reason == "max_timeout":
            terminal_component = (
                self._w("REWARD_TIGER_WIN")
                * self._w("MAX_TIMEOUT_SCALE")
            )
        elif reason == "repeat_timeout":
            terminal_component = (
                self._w("REWARD_TIGER_WIN")
                * self._w("REPEAT_STALL_SCALE")
            )
        else:
            # Non-terminal repeat discouragement:
            # quadratic in prior repeat count to ramp anti-cycling pressure.
            prev_count = int(info.get("repeat_prev_count", 0))
            if prev_count > 0:
                repeat_component = self._w("REWARD_REPEAT_STATE") * (prev_count ** 2)

        reward_total = (
            step_component
            + near_lock_component
            + block_component
            + bubble_component
            + cluster_component
            + center_component
            + goat_eaten_component
            + repeat_component
            + terminal_component
        )

        self._set_reward_components(
            info,
            reward_step_component=step_component,
            reward_near_lock_component=near_lock_component,
            reward_block_component=block_component,
            reward_bubble_component=bubble_component,
            reward_cluster_component=cluster_component,
            reward_center_component=center_component,
            reward_goat_eaten_component=goat_eaten_component,
            reward_repeat_component=repeat_component,
            reward_terminal_component=terminal_component,
            reward_decay_mult=decay_mult,
            reward_total=reward_total,
        )
        return reward_total


    def _sparse_reward_tiger(self, prev_obs, action, obs, terminated, truncated, info):
        """
        Compute the tiger-learner reward based on the transition summary stored in `info`.

        High-level composition:
        1. Base step penalty (discourages stalling)
        2. Capture rewards (positive reward for eating goats)
        3. Mobility shaping (reward when tiger move options increase)
        4. Positional shaping (center control during placing phase)
        5. Terminal rewards/penalties (win, loss, timeout)

        Important `info` fields consumed here:
        - reason, step_penalty, goats_eaten_this_turn
        - delta_moves, phase, center_score
        """

        # 1. Early exit / invalid action handling
        # These should almost never occur if action masking works correctly.
        # They act as safety guards in case an invalid action slips through.
        reason = info.get("reason")

        if reason == "already_terminated":
            return 0.0

        if reason == "invalid_soft":
            # Small penalty for illegal-but-non-fatal action
            return self._w("REWARD_INVALID_SOFT")

        if reason == "invalid_hard":
            # Large penalty for a serious invalid action that terminates the episode
            return abs(self._w("REWARD_INVALID_HARD"))

        # 2. Base step penalty
        # Small negative reward each turn to encourage faster wins
        # and discourage infinite wandering.
        shaped_reward = float(
            info.get("step_penalty", self._w("REWARD_STEP"))
        )

        # 3. Capture reward
        # Tigers receive positive reward when they capture goats.
        goats_eaten_this_turn = int(info.get("goats_eaten_this_turn", 0))

        if goats_eaten_this_turn > 0:
            shaped_reward += (
                self._w("REWARD_TIGER_CAPTURE")
                * goats_eaten_this_turn
            )

        # 4. Tiger mobility shaping
        # delta_moves measures change in number of available tiger moves.
        # Positive delta -> more mobility -> good for tigers.
        delta_moves = int(info.get("delta_moves", 0))

        if delta_moves != 0:
            shaped_reward += (
                abs(self._w("REWARD_BLOCK_TIGER"))
                * delta_moves
            )

        # 5. Center control shaping
        # Reward for occupying strategically strong center nodes. This
        # reward is only active in the placing phase of the game, so the
        # agent doesnt prioritize center control over eating goats.  
        phase = int(info.get("phase", self.phase))
        if phase == 0:
            shaped_reward += float(info.get("center_score", 0.0))

        # 6. Terminal outcome rewards / penalties
        # Large reward when tigers win the game.
        if reason in {"tiger_win_capture_threshold", "tiger_win_no_goat_moves"}:
            shaped_reward += (
                self._w("REWARD_TIGER_WIN_BONUS")
            )

        # Penalty when goats successfully immobilize the tigers.
        elif reason == "goat_win_no_tiger_moves":
            shaped_reward -= (
                self._w("REWARD_TIGER_LOSS_PENALTY")
            )

        # Timeout penalty (scaled loss)
        elif reason == "max_timeout":
            shaped_reward -= (
                self._w("REWARD_TIGER_LOSS_PENALTY")
                * self._w("MAX_TIMEOUT_SCALE")
            )

        return shaped_reward


    #  Group: Transition Engines
    #  Executes full goat-learner and tiger-learner turn transitions.
    #  Applies legality checks, terminal rules, and transition summaries.
    def _step_goat_transition(self, action):
        """
        Run one full goat-learner turn:
          1) Validate and apply goat action
          2) Recompute shaping metrics
          3) Let scripted tiger respond
          4) Finalize terminal checks and build info payload

        Notes:
        - This function returns base reward=0.0; shaping is applied later by
          `_apply_reward_fn(...)` using the `info` summary produced here.
        - `info["reason"]` is the canonical outcome marker used by reward code.
        """
        # Guard against stepping an already-terminal episode.
        if self.terminate:
            return self.get_state(), 0.0, True, False, self._build_step_info(reason="already_terminated")

        # Turn bookkeeping + action normalization for logging/debug.
        self.turns += 1
        flat_action = self._action_to_flat(action)
        decoded = self._safe_decode_action(action)
        base_info = {
            "goat_action": flat_action,
            "goat_decoded": decoded,
        }
        if decoded is None:
            info = self._build_step_info(reason="invalid_soft", invalid_action=True, **base_info)
            self.terminate = True
            return self.get_state(), 0.0, False, True, info

        # Parse validated action tuple.
        pos, dir_code = decoded
        goats_eaten_before = int(self.eaten)

        goat_desc = ""
        tiger_desc = ""

        if not (0 <= pos < BOARD_SIZE) or not (0 <= dir_code < DIR_CODES):
            info = self._build_step_info(reason="invalid_soft", invalid_action=True, **base_info)
            self.terminate = True
            return self.get_state(), 0.0, False, True, info

        # Snapshot shaping baselines before applying goat action.
        tiger_moves_before = int(self.prev_tiger_moves)
        bubble_before = int(self.prev_unreachable)
        spread_before = float(self.prev_tiger_spread)

        # Step penalty schedule:
        # - placing phase uses fixed REWARD_STEP
        # - moving phase ramps via MOVE_STEP_* knobs
        if self.phase == 0:
            step_penalty = self._w("REWARD_STEP")
        else:
            step_penalty = max(
                self._w("MOVE_STEP_BASE") + self._w("MOVE_STEP_SLOPE") * self.move_steps,
                self._w("MOVE_STEP_MIN"),
            )

        did_goat_move = False
        if self.phase == 0:
            # Placing-phase legality:
            # dir_code must be 0 and destination must be empty.
            if dir_code != 0 or self.board[pos] != 0:
                if DEBUG_INVALID:
                    print("how did you manage this error?", action)
                info = self._build_step_info(reason="invalid_soft", invalid_action=True, **base_info)
                self.terminate = True
                return self.get_state(), 0.0, False, True, info

            self.board[pos] = 1
            self.goats_placed += 1
            goat_desc = f"Goat placed at {idx_to_coord(pos)}"
            if self.goats_placed >= TOTAL_GOATS_TO_PLACE:
                # Transition from placing -> moving once all goats are placed.
                self.phase = 1
        else:
            # Moving-phase legality:
            # source must contain goat, destination must exist and be empty.
            dest = self.move_map.get(pos, {}).get(dir_code, None)
            if dest is None or self.board[pos] != 1 or self.board[dest] != 0:
                if DEBUG_INVALID:
                    print("Invalid goat move:", action)
                self.terminate = True
                info = self._build_step_info(
                    reason="invalid_hard",
                    invalid_action=True,
                    error="Invalid move",
                    **base_info,
                )
                return self.get_state(), 0.0, False, True, info

            self.board[pos] = 0
            self.board[dest] = 1
            did_goat_move = True
            goat_desc = f"Goat {idx_to_coord(pos)} -> {idx_to_coord(dest)}"

        if did_goat_move:
            self.move_steps += 1

        # Recompute tiger options immediately after goat move.
        tiger_options = self._tiger_moves(include_dir=False)
        tiger_moves_after = len(tiger_options)
        self.prev_tiger_moves = tiger_moves_after

        # Derived shaping signals consumed by goat reward:
        # near_lock, late-game factor, mobility delta, bubble delta, spread delta.
        near_lock = (0 < tiger_moves_after <= 2)
        late = min(
            max(self.turns - self._w("LATE_GAME_START_TURN"), 0)
            / (self._w("MAX_TURNS") - self._w("LATE_GAME_START_TURN")),
            1.0,
        )
        delta_moves = tiger_moves_before - tiger_moves_after

        bubble_after = self._compute_unreachable_safe_cells()
        self.prev_unreachable = bubble_after
        delta_bubble = bubble_after - bubble_before

        spread_after = self._tiger_spread()
        self.prev_tiger_spread = spread_after
        delta_spread = spread_before - spread_after

        # Center control score (sign/magnitude baked into knob constants).
        center_score = 0.0
        for idx in KEY_CENTERS:
            if self.board[idx] == 1:
                center_score += self._w("REWARD_CENTER_GOAT")
            elif self.board[idx] == 2:
                center_score += self._w("REWARD_CENTER_TIGER")

        common = dict(
            step_penalty=float(step_penalty),
            near_lock=bool(near_lock),
            late=float(late),
            delta_moves=int(delta_moves),
            delta_bubble=int(delta_bubble),
            delta_spread=float(delta_spread),
            center_score=float(center_score),
            goats_eaten_this_turn=0,
            repeat_prev_count=0,
            move_steps=int(self.move_steps),
        )

        # Immediate goat terminal win: tiger has no legal reply.
        if tiger_moves_after == 0:
            self.terminate = True
            self.last_move_desc = goat_desc or "Goat moved, then tigers had no moves"
            info = self._build_step_info(reason="goat_win_no_tiger_moves", winner="Goat", **base_info, **common)
            return self.get_state(), 0.0, True, False, info

        # Tiger response (scripted or learned model).
        tiger_options_with_dir = self._tiger_moves(include_dir=True)
        chosen = self._select_tiger_move(tiger_options, tiger_options_with_dir=tiger_options_with_dir)
        if chosen is None:
            if self.tiger_ai == TIGER_AI_MODEL:
                raise RuntimeError(
                    "Tiger model opponent returned no valid action. "
                    "Check tiger_model_predict_fn and mask handling."
                )
            raise RuntimeError(
                f"Tiger policy '{self.tiger_ai}' returned None despite "
                f"{len(tiger_options)} legal moves. State={self.get_state().tolist()}"
            )

        t_from, t_to, took_capture = chosen
        if took_capture:
            jumped_goat_pos = self._find_jumped_goat(t_from, t_to)
            if jumped_goat_pos is not None and self.board[jumped_goat_pos] == 1:
                self.board[jumped_goat_pos] = 0
                self.eaten += 1

        # Apply tiger board move.
        self.board[t_to] = 2
        self.board[t_from] = 0

        if t_from is not None and t_to is not None:
            if took_capture:
                tiger_desc = f"Tiger {idx_to_coord(t_from)} -> {idx_to_coord(t_to)} capture"
            else:
                tiger_desc = f"Tiger {idx_to_coord(t_from)} -> {idx_to_coord(t_to)}"

        goats_eaten_this_turn = int(self.eaten - goats_eaten_before)
        common["goats_eaten_this_turn"] = goats_eaten_this_turn
        common["tiger_move"] = (int(t_from), int(t_to), bool(took_capture))

        # Human-readable move summary used by render().
        if goat_desc and tiger_desc:
            self.last_move_desc = f"{goat_desc} | {tiger_desc}"
        elif goat_desc:
            self.last_move_desc = goat_desc
        elif tiger_desc:
            self.last_move_desc = tiger_desc
        else:
            self.last_move_desc = ""

        # Terminal checks after tiger response.
        if self.eaten >= GOATS_EATEN_FOR_TIGER_WIN:
            self.terminate = True
            info = self._build_step_info(reason="tiger_win_capture_threshold", winner="Tiger", **base_info, **common)
            return self.get_state(), 0.0, True, False, info

        # Global episode timeout.
        if self.turns >= self.max_turns:
            self.terminate = True
            info = self._build_step_info(reason="max_timeout", winner="MaxTimeout", **base_info, **common)
            return self.get_state(), 0.0, False, True, info

        # Repeat-state tracking for anti-cycling timeout logic.
        board_hash = (self.board.tobytes(), self.eaten, self.phase)
        prev_count = int(self.state_history.get(board_hash, 0))
        new_count = prev_count + 1
        self.state_history[board_hash] = new_count

        # Stall timeout if state repeats too often.
        if new_count >= self._w("MAX_REPEATS"):
            self.terminate = True
            common["repeat_prev_count"] = prev_count
            info = self._build_step_info(reason="repeat_timeout", winner="RepeatTimeout", **base_info, **common)
            return self.get_state(), 0.0, False, True, info

        # Non-terminal path: refresh goat legal moves and return transition summary.
        common["repeat_prev_count"] = prev_count
        self._update_valid_moves()
        info = self._build_step_info(**base_info, **common)
        return self.get_state(), 0.0, False, False, info

    def _step_tiger_transition(self, action):
        """
        Run one full tiger-learner turn:
          1) Validate and apply tiger action
          2) Compute tiger-facing shaping features
          3) Let goat opponent respond (model or random)
          4) Apply terminal checks and return transition summary

        Notes:
        - Like goat transition, this returns base reward=0.0 and relies on
          `_apply_reward_fn(...)` + `info` for final reward shaping.
        - Goat response policy is selected by `self.goat_opponent_ai`.
        """
        # Guard against stepping an already-terminal episode.
        if self.terminate:
            return self.get_state(), 0.0, True, False, self._build_step_info(reason="already_terminated")

        # Turn bookkeeping + action normalization for logging/debug.
        self.turns += 1
        flat_action = self._action_to_flat(action)
        decoded = self._safe_decode_action(action)
        base_info = {
            "goat_action": flat_action,
            "goat_decoded": decoded,
        }
        if decoded is None:
            info = self._build_step_info(reason="invalid_soft", invalid_action=True, **base_info)
            self.terminate = True
            return self.get_state(), 0.0, False, True, info

        # Parse validated action tuple.
        pos, dir_code = decoded
        goats_eaten_before = int(self.eaten)

        if not (0 <= pos < BOARD_SIZE) or not (0 <= dir_code < DIR_CODES):
            info = self._build_step_info(reason="invalid_soft", invalid_action=True, **base_info)
            self.terminate = True
            return self.get_state(), 0.0, False, True, info

        # Build legal tiger action table keyed by (from_pos, dir_code).
        # This ensures action IDs map deterministically to legal transitions.
        legal = {}
        for f, t, cap, d in self._tiger_moves(include_dir=True):
            key = (f, d)
            if key in legal:
                raise ValueError(f"duplicate tiger action key (from_pos,dir_code)={key}")
            legal[key] = (t, cap)

        if (pos, dir_code) not in legal:
            self.terminate = True
            info = self._build_step_info(
                reason="invalid_hard",
                invalid_action=True,
                error="Invalid tiger action",
                **base_info,
            )
            return self.get_state(), 0.0, False, True, info

        # Apply validated tiger move.
        dest, took_capture = legal[(pos, dir_code)]
        step_penalty = self._w("REWARD_STEP")

        self.board[dest] = 2
        self.board[pos] = 0

        if took_capture:
            jumped_goat_pos = self._find_jumped_goat(pos, dest)
            if jumped_goat_pos is not None and self.board[jumped_goat_pos] == 1:
                self.board[jumped_goat_pos] = 0
                self.eaten += 1

        # Tiger-facing shaping features:
        # capture count, mobility delta, and center score.
        goats_eaten_this_turn = int(self.eaten - goats_eaten_before)

        tiger_before = int(self.prev_tiger_moves)
        tiger_after = len(self._tiger_moves())
        self.prev_tiger_moves = tiger_after
        delta_moves = tiger_after - tiger_before

        center_score = 0.0
        for idx in KEY_CENTERS:
            if self.board[idx] == 2:
                center_score += abs(self._w("REWARD_CENTER_GOAT"))
            elif self.board[idx] == 1:
                center_score -= abs(self._w("REWARD_CENTER_GOAT"))

        common = dict(
            step_penalty=float(step_penalty),
            goats_eaten_this_turn=goats_eaten_this_turn,
            delta_moves=int(delta_moves),
            center_score=float(center_score),
            tiger_move=(int(pos), int(dest), bool(took_capture)),
        )

        # Immediate tiger terminal win by capture threshold.
        if self.eaten >= GOATS_EATEN_FOR_TIGER_WIN:
            self.terminate = True
            info = self._build_step_info(reason="tiger_win_capture_threshold", winner="Tiger", **base_info, **common)
            return self.get_state(), 0.0, True, False, info

        # Refresh goat legal moves before opponent response.
        self._update_valid_moves()

        # Goat-opponent helper stack:
        # - goat_mask: legal action mask for current board
        # - goat_random_action: uniform legal sample
        # - goat_model_action: model prediction with legality validation
        def goat_mask():
            """Build a legal-action mask for the scripted/model goat opponent."""
            mask = np.zeros(BOARD_SIZE * DIR_CODES, dtype=bool)
            for mv in self.valid_moves:
                if np.isscalar(mv):
                    flat = int(mv)
                else:
                    flat = self.encode_action(mv[0], mv[1])
                if 0 <= flat < mask.size:
                    mask[flat] = True
            return mask

        def goat_random_action():
            """Sample one legal goat action uniformly from the current mask."""
            mask = goat_mask()
            legal_idxs = np.flatnonzero(mask)
            if legal_idxs.size == 0:
                return None
            flat = int(np.random.choice(legal_idxs))
            return flat // DIR_CODES, flat % DIR_CODES

        def goat_model_action():
            """Query goat model policy and return a validated decoded action."""
            if self._goat_model_predict_fn is None:
                return None
            mask = goat_mask()
            obs = self.get_state()
            try:
                flat = int(self._goat_model_predict_fn(obs, mask))
                if 0 <= flat < mask.size and mask[flat]:
                    return flat // DIR_CODES, flat % DIR_CODES
            except Exception:
                return None
            return None

        # Prefer model goat when configured; fallback to random if unavailable
        # or if model proposes an invalid action.
        goat_choice = None
        # Model goat is optional; invalid/missing model prediction falls back to random goat.
        if self.goat_opponent_ai == GOAT_AI_MODEL:
            goat_choice = goat_model_action()
        if goat_choice is None:
            goat_choice = goat_random_action()

        # No legal goat action -> tiger terminal win.
        if goat_choice is None:
            self.terminate = True
            info = self._build_step_info(reason="tiger_win_no_goat_moves", winner="Tiger", **base_info, **common)
            return self.get_state(), 0.0, True, False, info

        # Apply goat response using phase-appropriate legality.
        g_pos, g_dir = goat_choice
        if self.phase == 0:
            if self.board[g_pos] == 0 and g_dir == 0:
                self.board[g_pos] = 1
                self.goats_placed += 1
                if self.goats_placed >= TOTAL_GOATS_TO_PLACE:
                    self.phase = 1
        else:
            g_dest = self.move_map[g_pos].get(g_dir, None)
            if g_dest is not None and self.board[g_pos] == 1 and self.board[g_dest] == 0:
                self.board[g_pos] = 0
                self.board[g_dest] = 1
                self.move_steps += 1

        # Post-response terminal checks.
        if len(self._tiger_moves()) == 0:
            self.terminate = True
            info = self._build_step_info(reason="goat_win_no_tiger_moves", winner="Goat", **base_info, **common)
            return self.get_state(), 0.0, True, False, info

        # Global episode timeout.
        if self.turns >= self.max_turns:
            self.terminate = True
            info = self._build_step_info(reason="max_timeout", winner="MaxTimeout", **base_info, **common)
            return self.get_state(), 0.0, False, True, info

        # Non-terminal path: refresh masks and return transition summary.
        self._update_valid_moves()
        info = self._build_step_info(**base_info, **common)
        return self.get_state(), 0.0, False, False, info


    def _find_jumped_goat(self, t_from: int, t_to: int):
        """
        Given a tiger capture move t_from -> t_to (which must be a jump),
        find the intermediate neighbor that was jumped over.
        Returns jumped_pos or None if not found.
        """
        # Special hub rule for node 0
        if t_from == 0:
            for dir_code, neigh in self.move_map[0].items():
                if neigh is None:
                    continue
                jump = HUB0_CAPTURE_JUMP.get(dir_code, None)
                if jump == t_to:
                    return neigh
            return None

        # Default rule (same dir_code twice)
        for dir_code, neigh in self.move_map.get(t_from, {}).items():
            if neigh is None:
                continue
            jump = self.move_map.get(neigh, {}).get(dir_code, None)
            if jump == t_to:
                return neigh
        return None
    


    #  Group: Tiger Policy Selection
    #  Selects tiger responses for goat-learner turns.
    #  Contains smart heuristic and greedy capture-biased policies.
    def _select_tiger_move(self, tiger_options, tiger_options_with_dir=None):
        """Select tiger move using configured tiger policy (smart, greedy, or model)."""
        if self.learner_role == GOAT_LEARNER:
            # tiger_options here should be 3-tuples
            if tiger_options and len(tiger_options[0]) != 3:
                raise ValueError("Goat-learner tiger_options must be (from,to,cap) 3-tuples")

        if self.tiger_ai == TIGER_AI_MODEL:
            return self._model_tiger_move(tiger_options_with_dir or [])
        if self.tiger_ai == TIGER_AI_SMART:
            return self._smart_tiger_move(tiger_options)
        return self._greedy_tiger_move(tiger_options)

    def _model_tiger_move(self, tiger_options_with_dir):
        """
        Query tiger model opponent and map predicted flat action to a legal tiger move.

        Expected callback signature:
          tiger_model_predict_fn(obs, mask) -> flat action id
        """
        if self._tiger_model_predict_fn is None:
            return None
        if not tiger_options_with_dir:
            return None

        legal = {}
        mask = np.zeros(BOARD_SIZE * DIR_CODES, dtype=bool)
        for from_pos, to_pos, cap, dir_code in tiger_options_with_dir:
            flat = self.encode_action(from_pos, dir_code)
            legal[flat] = (from_pos, to_pos, cap)
            mask[flat] = True

        try:
            flat = int(self._tiger_model_predict_fn(self.get_state(), mask))
        except Exception:
            return None

        return legal.get(flat, None)

    def _smart_tiger_move(self, tiger_options, debug: bool = False):
        """
        Pick a heuristic smart tiger move from legal options.
          1) Take captures first
          2) Try to occupy anchor positions: 0 and one of {9, 10}
          3) Otherwise roam toward stronger board squares
          4) Fallback to any legal move
        """
        board = self.board
        move_map = self.move_map
        tiger_moves = tiger_options

        POS_VALS = {
            0: 8, 1: 4, 2: 6, 3: 7, 4: 7, 5: 6, 6: 4, 7: 4, 8: 7, 9: 8, 10: 8,
            11: 7, 12: 4, 13: 4, 14: 6, 15: 7, 16: 7, 17: 6, 18: 4, 19: 4,
            20: 5, 21: 5, 22: 4,
        }

        def debug_print(*args):
            """Only print diagnostics when debug mode is on."""
            if debug:
                print("[TIGER DEBUG]", *args)

        def is_high_value(pos):
            """Return True if a square is considered worth roaming toward."""
            return POS_VALS[pos] >= 5

        def occupied_by_tiger(pos):
            """Small helper for tiger occupancy checks."""
            return board[pos] == 2

        def bfs_find_next_step(start_pos, target_positions):
            """
            Return the first step on a shortest path from start_pos to any target.
            Returns (first_step, distance), or (None, None) if unreachable.
            """
            if start_pos in target_positions:
                return (None, 0)

            queue = []
            visited = {start_pos}

            # Seed BFS with immediate legal walkable neighbors
            for _, neighbor in move_map[start_pos].items():
                if neighbor is None:
                    continue

                is_walkable = (board[neighbor] == 0) or (neighbor in target_positions)
                if not is_walkable:
                    continue

                if neighbor in target_positions:
                    return (neighbor, 1)

                queue.append((neighbor, neighbor, 1))  # (current, first_step, distance)
                visited.add(neighbor)

            # Expand outward until a target is found
            while queue:
                curr, first_step, dist = queue.pop(0)

                if curr in target_positions:
                    return (first_step, dist)

                for _, neighbor in move_map[curr].items():
                    if neighbor is None or neighbor in visited:
                        continue

                    is_walkable = (board[neighbor] == 0) or (neighbor in target_positions)
                    if not is_walkable:
                        continue

                    visited.add(neighbor)
                    queue.append((neighbor, first_step, dist + 1))

            return (None, None)

        # MAIN LOGIC
        if not tiger_moves:
            debug_print("No legal tiger moves.")
            return None

        debug_print("Board:", board)
        debug_print("Legal tiger moves:", tiger_moves)

        # --- 1) CAPTURE LOGIC ---
        captures = [move for move in tiger_moves if move[2]]
        has_capture = len(captures) > 0

        if has_capture:
            random.shuffle(captures)  # break ties a little before max()
            best_capture = max(captures, key=lambda move: POS_VALS[move[1]])

            debug_print(
                "Capture phase:",
                "candidate captures =", captures,
                "chosen =", best_capture,
                "dest_pos_val =", POS_VALS[best_capture[1]],
            )
            return best_capture

        # --- 2) ANCHOR LOGIC ---
        # The tiger strategy prefers to control two important board regions:
        #   * Anchor 0 (top of the board)
        #   * One of the central anchors {9, 10}
        #
        # These positions tend to give the tigers better mobility and capture lanes.
        # If either anchor is not currently occupied by a tiger, we attempt to move
        # the closest tiger toward that anchor using BFS pathfinding.

        current_tiger_positions = [i for i in range(23) if occupied_by_tiger(i)]

        anchor_0_occupied = occupied_by_tiger(0)
        anchor_primary_occupied = occupied_by_tiger(9) or occupied_by_tiger(10)

        # Determine which anchors we still need to claim
        need_anchor0 = not anchor_0_occupied
        need_primary = not anchor_primary_occupied

        debug_print(
            "Anchor status:",
            f"anchor_0_occupied={anchor_0_occupied}, "
            f"anchor_primary_occupied={anchor_primary_occupied}, "
            f"need_anchor0={need_anchor0}, need_primary={need_primary}",
        )

        if need_anchor0 or need_primary:

            # Build a list of anchor targets that we want to move toward
            # Each entry is (group_name, list_of_target_positions)
            anchor_groups = []

            if need_anchor0:
                anchor_groups.append(("secondary", [0]))

            if need_primary:
                # Prefer moving to an empty primary anchor
                primary_targets = [p for p in (9, 10) if board[p] == 0]

                # If both are occupied (rare but possible during transitions),
                # we still treat them as targets so BFS can plan toward them.
                if not primary_targets:
                    primary_targets = [9, 10]

                anchor_groups.append(("primary", primary_targets))

            debug_print("Anchor groups:", anchor_groups)

            # Track the best candidate anchor move
            # Structure: (distance, tiger_start_pos, first_step, group_name, targets)
            best = None

            for t_pos in current_tiger_positions:

                # If anchor 0 is already held, we avoid moving that tiger away
                if anchor_0_occupied and t_pos == 0:
                    debug_print("Skipping tiger at 0 (already satisfying anchor 0).")
                    continue

                # Same idea for the primary anchors
                if anchor_primary_occupied and t_pos in (9, 10):
                    debug_print(f"Skipping tiger at {t_pos} (already satisfying primary anchor).")
                    continue

                # Try finding a path from this tiger to each anchor group
                for group_name, targets in anchor_groups:

                    step, dist = bfs_find_next_step(t_pos, targets)

                    # If BFS can't reach the anchor through empty spaces, skip
                    if step is None:
                        continue

                    debug_print(
                        f"Anchor candidate: tiger at {t_pos} -> step {step} "
                        f"(group {group_name}, targets {targets}, dist {dist})"
                    )

                    # Choose the tiger that can reach an anchor in the fewest steps
                    if (best is None) or (dist < best[0]):
                        best = (dist, t_pos, step, group_name, targets)

            # Convert the BFS suggestion into a legal tiger move
            if best is not None:
                dist, from_pos, step, group_name, targets = best
                chosen_move = None
                for move in tiger_moves:
                    if move[0] == from_pos and move[1] == step:
                        chosen_move = move
                        break

                if chosen_move is not None:
                    debug_print(
                        "Anchor choice:",
                        f"group={group_name}, targets={targets}, dist={dist},",
                        f"from={from_pos}, to={step}, move={chosen_move}",
                    )
                    return chosen_move

                debug_print(
                    "Anchor choice failed: BFS suggested step not in tiger_moves.",
                    f"group={group_name}, targets={targets}, dist={dist},",
                    f"from={from_pos}, to={step}",
                )
            else:
                debug_print("Anchor logic: no reachable anchor path found, falling through.")

        # --- 3) ROAMING LOGIC ---
        # If no captures or anchor moves are available, the tiger enters a
        # "roaming" mode. The goal here is simple positional improvement:
        #
        #   * Move toward high-value board squares (POS_VALS >= 5)
        #   * Avoid abandoning important anchor positions
        #
        # This encourages the tiger to drift toward strong control squares
        # instead of wandering randomly.

        high_value_moves = []

        for move in tiger_moves:
            t_from, t_to, _ = move

            # Never abandon anchor 0 once we control it
            # This keeps one tiger guarding the top anchor
            if t_from == 0:
                continue

            # Prevent leaving the only occupied primary anchor (9 or 10)
            # If exactly one tiger is holding the central anchor region,
            # we keep it there to maintain positional control.
            leaving_only_primary_anchor = (
                anchor_primary_occupied
                and (
                    (t_from == 9 and board[9] == 2 and board[10] != 2)
                    or (t_from == 10 and board[10] == 2 and board[9] != 2)
                )
            )

            if leaving_only_primary_anchor:
                continue

            # If the destination square has strong positional value,
            # consider it as a roaming candidate
            if is_high_value(t_to):
                high_value_moves.append(move)

        # Choose randomly among strong roaming options
        # (adds slight unpredictability)
        if high_value_moves:
            chosen = random.choice(high_value_moves)

            debug_print(
                "Roaming phase:",
                "high_value_moves =", high_value_moves,
                "chosen =", chosen,
                "dest_pos_val =", POS_VALS[chosen[1]],
            )

            return chosen

        # --- 4) FALLBACK ---
        fallback = random.choice(tiger_moves)
        debug_print(
            "Fallback phase: no captures, no anchor moves, no high-value roam.",
            "Fallback chosen move =", fallback,
        )
        return fallback

    def _greedy_tiger_move(self, tiger_options):
        """Pick a capture-biased random tiger move from legal options."""
        captures = [m for m in tiger_options if m[2]]
        capture_bias = self._w("BASE_TIGER_CAPTURE_BIAS")

        if captures and random.random() < capture_bias:
            return random.choice(captures)

        if tiger_options:
            return random.choice(tiger_options)

        return None
    


    #  Group: Move Generation and Mask Sources
    #  Enumerates legal moves used by masking, transitions, and opponents.
    #  Serves as the source of truth for move legality.
    def _tiger_moves(self, include_dir: bool = False):
        """
        Compute all legal tiger moves.

        Returns:
          list of (from_pos, to_pos, is_capture[, dir_code])

        Deterministic rule:
          - dir_code is always the actual move_map direction used for both:
              (from -> neighbor) and (neighbor -> jump)
          - No special-case direction remapping for node 0.
        """
        moves = []

        for i in range(BOARD_SIZE):
            if self.board[i] != 2:
                continue

            for dir_code, neigh in self.move_map.get(i, {}).items():
                if neigh is None:
                    continue

                # normal move
                if self.board[neigh] == 0:
                    moves.append((i, neigh, False, dir_code) if include_dir else (i, neigh, False))
                    continue

                # capture candidate: goat in neighbor
                if self.board[neigh] != 1:
                    continue

                # Hub constraint from the original x96_TnG.ipynb rules:
                # row-1 nodes (b1..e1) cannot capture "up" through b0.
                if i in (2, 3, 4, 5) and dir_code == 1:
                    continue

                # jump destination
                if i == 0:
                    jump = HUB0_CAPTURE_JUMP.get(dir_code, None)
                else:
                    jump = self.move_map.get(neigh, {}).get(dir_code, None)

                if jump is None or self.board[jump] != 0:
                    continue

                moves.append((i, jump, True, dir_code) if include_dir else (i, jump, True))

        return moves
    def _update_valid_moves(self):
        """
        Recompute all legal goat actions and store in self.valid_moves.

        - During placing phase: any empty cell with dir_code=0
        - During moving phase: goats can move to empty neighbors
        """
        new_moves = []

        if self.phase == 0:
            # placing phase: any empty cell is valid
            for i in range(BOARD_SIZE):
                if self.board[i] == 0:
                    new_moves.append(self.encode_action(i, 0))
        else:
            # moving phase: move goats along legal directions
            for i in range(BOARD_SIZE):
                if self.board[i] == 1:  # goat
                    for dir_code, dest in self.move_map[i].items():
                        if dest is not None and self.board[dest] == 0:
                            new_moves.append(self.encode_action(i, dir_code))

        self.valid_moves = new_moves


    #  Group: Board Metrics and Graph Utilities
    #  Computes reachability and geometry features for shaping signals.
    #  Includes shortest-path helpers on the static board graph.

    def _compute_unreachable_safe_cells(self) -> int:
        """
        Count how many cells are unreachable by any tiger if tigers
        can only walk through empty cells and onto empty cells.
        (Goats act as walls for this reachability notion.)

        This approximates "bubbles": safe pockets tigers can't reach.
        """
        # 1) Find starting tiger positions
        tiger_starts = [i for i in range(BOARD_SIZE) if self.board[i] == 2]

        if not tiger_starts:
            # no tigers: treat all empty/goat cells as unreachable (safe)
            return int(np.sum((self.board == 0) | (self.board == 1)))
        
        # 2) BFS from all tigers through empty cells only
        visited = self._visited_buffer
        visited[:] = False

        q = deque()
        for t in tiger_starts:
            visited[t] = True
            q.append(t)

        while q:
            cur = q.popleft()
            for _, dest in self.move_map.get(cur, {}).items():
                # tiger can walk into empty neighbor (ignore goats in reachability)
                if self.board[dest] == 0 and not visited[dest]:
                    visited[dest] = True
                    q.append(dest)

        # 3) Count unreachable cells that are useful to goats (empty or goat)
        unreachable = 0
        for i in range(BOARD_SIZE):
            if not visited[i] and self.board[i] in (0, 1):
                unreachable += 1
        
        return unreachable
    

    def _precompute_all_pairs_dist(self) -> np.ndarray:
        """
        Precompute BFS shortest path lengths between all board positions
        in the static graph defined by move_map, ignoring pieces.
        """
        dist = np.full((BOARD_SIZE, BOARD_SIZE), BOARD_SIZE, dtype=np.int8)

        for start in range(BOARD_SIZE):
            visited = np.zeros(BOARD_SIZE, dtype=bool)
            q = deque()
            q.append((start, 0))
            visited[start] = True
            dist[start, start] = 0

            while q:
                cur, d = q.popleft()
                for _, dest in self.move_map.get(cur, {}).items():
                    if not visited[dest]:
                        visited[dest] = True
                        dist[start, dest] = d + 1
                        q.append((dest, d + 1))

        return dist
    

    def _shortest_path_len(self, start: int, goal: int) -> int:
        """
        Shortest path length in the static board graph using precomputed
        all-pairs distances.
        """
        return int(self._dist_matrix[start, goal])
    
    
    # we get a positive reward if tiger spread decreases
    def _tiger_spread(self) -> float:
        """
        Sum of pairwise distances between tigers.
        Smaller => tigers more clustered / forced together.
        """
        tiger_positions = [i for i in range(BOARD_SIZE) if self.board[i] == 2]
        if len(tiger_positions) < 2:
            return 0.0

        total = 0.0
        for idx_a in range(len(tiger_positions)):
            for idx_b in range(idx_a + 1, len(tiger_positions)):
                a = tiger_positions[idx_a]
                b = tiger_positions[idx_b]
                total += self._shortest_path_len(a, b)

        return float(total)
# end class TnGEnv()

<a id="training-helpers"></a>

## Training helpers
These definitions validate settings, create masked environments, log outcomes and
reward components, and save models. No training starts until **Run training**.

The environment wrapper supplies action masks and, when mixing is enabled, samples
the opponent at each reset. `DummyVecEnv` runs the configured environments
sequentially. The callback retains cumulative/rolling outcome rates, opponent-mix
statistics, episode rewards, and available reward-component metrics in TensorBoard.
Reward-component coverage follows the environment's learner-specific `info` fields.
Each phase gets its own logs and saved settings; the environment is closed after use.


In [4]:
SOURCE_COMMIT = "1b8ba721253530dad0be529a21628353b3db59df" # for metadata, may shorten to human readable tag later.
PHASE_KEYS = {"timesteps", "opponent", "mix_prob", "checkpoints_per_run", "goat_model_path", "reward_overrides"}

class WinStatsCallback(BaseCallback):
    """
    Windowed win/timeout stats + optional realized tiger-mode tracking.
    If env includes info["tiger_ai"] at episode end, we will track it.
    """
    def __init__(self, log_every_episodes: int = 200, window_episodes: int = 1_000, verbose: int = 0):
        super().__init__(verbose)
        self.log_every_episodes = log_every_episodes
        self.last_logged_episodes = 0
        self.window_episodes = window_episodes

        # cumulative
        self.episodes = 0
        self.goat_wins = 0
        self.tiger_wins = 0
        self.max_timeouts = 0
        self.repeat_timeouts = 0
        self.window_snapshots = [] # Reserved for windowed-stat snapshots; currently unused

        # rolling window of winners
        self.window = deque(maxlen=window_episodes)

        # rolling window of realized opponent modes (optional)
        self.window_tiger_ai = deque(maxlen=window_episodes)  # values: greedy/smart/None
        self.window_goat_ai = deque(maxlen=window_episodes)   # values: random/model/None
        self.reward_component_names = [
            "reward_step_component",
            "reward_near_lock_component",
            "reward_block_component",
            "reward_bubble_component",
            "reward_cluster_component",
            "reward_center_component",
            "reward_goat_eaten_component",
            "reward_repeat_component",
            "reward_terminal_component",
            "reward_decay_mult",
            "reward_total",
        ]
        self.reward_component_windows = {
            name: deque(maxlen=1_000) for name in self.reward_component_names
        }
        self.episode_reward_window = deque(maxlen=window_episodes)
        self.terminal_penalty_windows = {
            "Goat": deque(maxlen=window_episodes),
            "Tiger": deque(maxlen=window_episodes),
            "MaxTimeout": deque(maxlen=window_episodes),
            "RepeatTimeout": deque(maxlen=window_episodes),
        }

    def _on_step(self) -> bool:
        dones = self.locals.get("dones", None)
        infos = self.locals.get("infos", None)

        if infos is not None:
            for info in infos:
                if info is None:
                    continue
                for name in self.reward_component_names:
                    if name in info:
                        self.reward_component_windows[name].append(float(info.get(name, 0.0)))

        if dones is not None and infos is not None:
            for done, info in zip(dones, infos):
                if not done:
                    continue

                winner = info.get("winner", None)

                self.episodes += 1
                if winner == "Goat":
                    self.goat_wins += 1
                elif winner == "Tiger":
                    self.tiger_wins += 1
                elif winner == "MaxTimeout":
                    self.max_timeouts += 1
                elif winner == "RepeatTimeout":
                    self.repeat_timeouts += 1

                self.window.append(winner)

                # optional: realized opponent mode
                # TnGEnv in the environment cell supplies the episode's tiger mode in info["tiger_ai"].
                self.window_tiger_ai.append(info.get("tiger_ai", None))
                self.window_goat_ai.append(info.get("goat_opponent_ai", info.get("goat_ai", None)))

                episode_info = info.get("episode", None)
                if isinstance(episode_info, dict) and "r" in episode_info:
                    self.episode_reward_window.append(float(episode_info["r"]))

                if winner in self.terminal_penalty_windows:
                    self.terminal_penalty_windows[winner].append(
                        float(info.get("reward_terminal_component", 0.0))
                    )

        # always keep current episode index in logs
        self.logger.record("1.episode_stats (total episodes)/1.episode_index", float(self.episodes))

        if (self.episodes - self.last_logged_episodes) >= self.log_every_episodes and self.episodes > 0:
            total = float(self.episodes)

            self.last_logged_episodes = self.episodes
            # Order of logs (to control TB display grouping)
            self.logger.record("1.episode_stats (total episodes)/2.goat_win_rate", self.goat_wins / total)
            self.logger.record("1.episode_stats (total episodes)/3.tiger_win_rate", self.tiger_wins / total)
            self.logger.record("1.episode_stats (total episodes)/4.max_timeout_rate", self.max_timeouts / total)
            self.logger.record("1.episode_stats (total episodes)/5.repeat_timeout_rate", self.repeat_timeouts / total)

            w = len(self.window)
            if w > 0:
                goat = sum(1 for x in self.window if x == "Goat") / w
                tiger = sum(1 for x in self.window if x == "Tiger") / w
                to = sum(1 for x in self.window if x == "MaxTimeout") / w
                st = sum(1 for x in self.window if x == "RepeatTimeout") / w

                self.logger.record("2.window_stats (per 1000 eps)/1.episodes_tracked", float(w))
                self.logger.record("2.window_stats (per 1000 eps)/2.goat_win_rate", goat)
                self.logger.record("2.window_stats (per 1000 eps)/3.tiger_win_rate", tiger)
                self.logger.record("2.window_stats (per 1000 eps)/4.max_timeout_rate", to)
                self.logger.record("2.window_stats (per 1000 eps)/5.repeat_timeout_rate", st)

                # optional realized opponent distribution in the last-N window
                smart = 0
                greedy = 0
                for v in self.window_tiger_ai:
                    if v == TIGER_AI_SMART or v == "smart" or v == "SMART":
                        smart += 1
                    elif v == TIGER_AI_GREEDY or v == "greedy" or v == "GREEDY":
                        greedy += 1
                denom = smart + greedy
                if denom > 0:
                    self.logger.record("2.window_stats (per 1000 eps)/6.realized_greedy_tiger_frac", greedy / denom)
                    self.logger.record("2.window_stats (per 1000 eps)/7.realized_smart_tiger_frac", smart / denom)

                model = 0
                random_goat = 0
                for v in self.window_goat_ai:
                    if v == GOAT_AI_MODEL or v == "model" or v == "MODEL":
                        model += 1
                    elif v == GOAT_AI_RANDOM or v == "random" or v == "RANDOM":
                        random_goat += 1
                denom = model + random_goat
                if denom > 0:
                    self.logger.record("2.window_stats (per 1000 eps)/8.realized_random_goat_frac", random_goat / denom)
                    self.logger.record("2.window_stats (per 1000 eps)/9.realized_model_goat_frac", model / denom)

            reward_component_labels = {
                "reward_step_component": "1.step_penalty_mean",
                "reward_near_lock_component": "2.near_lock_mean",
                "reward_block_component": "3.block_mean",
                "reward_bubble_component": "4.bubble_mean",
                "reward_cluster_component": "5.cluster_mean",
                "reward_center_component": "6.center_mean",
                "reward_goat_eaten_component": "7.goat_eaten_mean",
                "reward_repeat_component": "8.repeat_mean",
                "reward_terminal_component": "9.terminal_mean",
                "reward_decay_mult": "10.decay_mult_mean",
                "reward_total": "11.total_reward_mean",
            }
            for name, label in reward_component_labels.items():
                values = self.reward_component_windows[name]
                if values:
                    self.logger.record(
                        f"4.reward_components (per 1000 steps)/{label}",
                        sum(values) / float(len(values)),
                    )

            if self.episode_reward_window:
                self.logger.record(
                    "5.episode_reward (per 1000 eps)/1.episode_reward_mean",
                    sum(self.episode_reward_window) / float(len(self.episode_reward_window)),
                )

            terminal_labels = {
                "Goat": "2.goat_terminal_reward_mean",
                "Tiger": "3.tiger_terminal_penalty_mean",
                "MaxTimeout": "4.max_timeout_terminal_penalty_mean",
                "RepeatTimeout": "5.repeat_timeout_terminal_penalty_mean",
            }
            for outcome, label in terminal_labels.items():
                values = self.terminal_penalty_windows[outcome]
                if values:
                    self.logger.record(
                        f"5.episode_reward (per 1000 eps)/{label}",
                        sum(values) / float(len(values)),
                    )

        return True
    


def positive_int(value, label, minimum=1):
    if isinstance(value, bool) or not isinstance(value, int) or value < minimum:
        raise ValueError(f"{label} must be an integer >= {minimum}.")


def finite_number(value, label, low=None, high=None):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        raise ValueError(f"{label} must be a finite number.")
    if (low is not None and value < low) or (high is not None and value > high):
        raise ValueError(f"{label} must be between {low} and {high}.")


def checked_model_path(value, label):
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{label} requires an existing .zip model path.")
    path = Path(value).expanduser()
    if path.suffix.lower() != ".zip" or not path.is_file():
        raise ValueError(f"{label}: model file not found: {path}")
    return str(path.resolve())


def validate_common():
    if LEARNER_ROLE not in {"goat", "tiger"}:
        raise ValueError('LEARNER_ROLE must be "goat" or "tiger".')
    if not isinstance(EXPERIMENT_NAME, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", EXPERIMENT_NAME):
        raise ValueError("EXPERIMENT_NAME must start with a letter/number and use only letters, numbers, _, -, or .")
    if not isinstance(OUTPUT_ROOT, str) or not OUTPUT_ROOT.strip():
        raise ValueError("OUTPUT_ROOT must be a nonempty path.")
    if DEVICE not in {"cpu", "cuda"}:
        raise ValueError('DEVICE must be "cpu" or "cuda".')
    if DEVICE == "cuda" and not torch.cuda.is_available():
        raise ValueError('CUDA is unavailable. Select a GPU runtime or set DEVICE = "cpu".')


def validate_phase(phase, role):
    positive_int(phase["timesteps"], "timesteps")
    positive_int(phase["checkpoints_per_run"], "checkpoints_per_run", 0)
    allowed = {"greedy", "smart"} if role == "goat" else {"random", "model"}
    if phase["opponent"] not in allowed:
        raise ValueError(f"{role} learner opponent must be one of {sorted(allowed)}.")
    mix = phase["mix_prob"]
    if mix is not None:
        finite_number(mix, "mix_prob", 0, 1)
    rewards = phase["reward_overrides"]
    if not isinstance(rewards, dict):
        raise ValueError("reward_overrides must be a dictionary.")
    unknown = set(rewards) - set(DEFAULT_KNOBS)
    if unknown:
        raise ValueError(f"Unknown reward/game settings: {sorted(unknown)}")
    for key, value in rewards.items():
        finite_number(value, key)
        if key in {"MAX_TURNS", "MAX_REPEATS"}:
            positive_int(value, key)
    needs_model = role == "tiger" and ((mix is None and phase["opponent"] == "model") or (mix is not None and mix > 0))
    if needs_model:
        phase["goat_model_path"] = checked_model_path(phase["goat_model_path"], "goat_model_path")
    return phase


def base_phase():
    return dict(timesteps=TIMESTEPS, opponent=OPPONENT, mix_prob=MIX_PROB,
                checkpoints_per_run=CHECKPOINTS_PER_RUN, goat_model_path=GOAT_MODEL_PATH,
                reward_overrides=copy.deepcopy(REWARD_OVERRIDES))


def merge_phase(base, overrides):
    if not isinstance(overrides, dict) or set(overrides) - PHASE_KEYS:
        raise ValueError(f"Phase overrides must be a dictionary with keys: {sorted(PHASE_KEYS)}")
    result = copy.deepcopy(base)
    if not isinstance(result["reward_overrides"], dict) or not isinstance(overrides.get("reward_overrides", {}), dict):
        raise ValueError("reward_overrides must be a dictionary.")
    result.update({k: v for k, v in overrides.items() if k != "reward_overrides"})
    result["reward_overrides"].update(overrides.get("reward_overrides", {}))
    return result


def normalized_suite():
    variations = {"baseline": None} if VARIATIONS is None else VARIATIONS
    if not isinstance(variations, dict) or not variations:
        raise ValueError("VARIATIONS must be None or a nonempty dictionary.")
    suite = {}
    for name, spec in variations.items():
        if not isinstance(name, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", name):
            raise ValueError("Variation names must use letters, numbers, _, -, or . and start with a letter/number.")
        base = base_phase()
        if spec is None:
            phases = [{}]
        elif isinstance(spec, list):
            phases = spec
        elif isinstance(spec, dict):
            if "phases" in spec:
                base = merge_phase(base, {k: v for k, v in spec.items() if k != "phases"})
                phases = spec["phases"]
            else:
                phases = [spec]
        else:
            raise ValueError(f"Invalid variation: {name}")
        if not isinstance(phases, list) or not phases:
            raise ValueError(f"{name}: phases must be a nonempty list.")
        suite[name] = [validate_phase(merge_phase(base, phase), LEARNER_ROLE) for phase in phases]
    return suite


def validate_training():
    validate_common()
    for name, value in [("NUM_ENVS", NUM_ENVS), ("N_STEPS", N_STEPS), ("N_EPOCHS", N_EPOCHS)]:
        positive_int(value, name)
    positive_int(SEED, "SEED", 0)
    if SEED + NUM_ENVS > 2**32:
        raise ValueError("SEED + NUM_ENVS must be <= 2**32.")
    if RESUME_MODEL_PATH is not None:
        checked_model_path(RESUME_MODEL_PATH, "RESUME_MODEL_PATH")
    else:
        positive_int(BATCH_SIZE, "BATCH_SIZE", 2)
        if N_STEPS * NUM_ENVS < BATCH_SIZE or (N_STEPS * NUM_ENVS) % BATCH_SIZE:
            raise ValueError("BATCH_SIZE must divide N_STEPS * NUM_ENVS and not exceed it.")
        finite_number(LEARNING_RATE, "LEARNING_RATE", 0)
        if LEARNING_RATE == 0:
            raise ValueError("LEARNING_RATE must be positive.")
        finite_number(GAMMA, "GAMMA", 0, 1)
        finite_number(ENT_COEF, "ENT_COEF", 0)
        if not isinstance(NET_ARCH, list) or not NET_ARCH:
            raise ValueError("NET_ARCH must be a nonempty list of positive widths.")
        for width in NET_ARCH:
            positive_int(width, "NET_ARCH width")
    return normalized_suite()


class NotebookMatchup(Monitor):
    """Resample opponents at reset using TnGEnv from the environment definition cell."""
    def __init__(self, env, role, mix):
        super().__init__(env)
        self.role, self.mix = role, mix

    def reset(self, **kwargs):
        if self.mix is not None:
            selected = random.random() < self.mix
            if self.role == "goat":
                self.env.tiger_ai = "smart" if selected else "greedy"
            else:
                self.env.goat_opponent_ai = "model" if selected else "random"
        return super().reset(**kwargs)


def make_notebook_env(phase, role, device):
    mix = phase["mix_prob"]
    needs_model = role == "tiger" and ((mix is None and phase["opponent"] == "model") or (mix is not None and mix > 0))
    goat_model = MaskablePPO.load(phase["goat_model_path"], device=device) if needs_model else None

    def predict_goat(obs, mask):
        action, _ = goat_model.predict(np.asarray(obs), deterministic=True, action_masks=np.asarray(mask, dtype=bool))
        return int(np.asarray(action).item())

    # Mixing selects the actual opponent at every reset, including the first one.
    env = TnGEnv(learner_role=role, reward_weights=phase["reward_overrides"],
                 tiger_ai=phase["opponent"] if role == "goat" else "smart",
                 goat_opponent_ai=phase["opponent"] if role == "tiger" else "random",
                 goat_model_predict_fn=predict_goat if needs_model else None)
    if goat_model is not None and (goat_model.observation_space != env.observation_space or goat_model.action_space != env.action_space):
        env.close()
        raise ValueError("Goat opponent model has incompatible observation/action spaces.")
    return ActionMasker(NotebookMatchup(env, role, mix), lambda wrapped: wrapped.unwrapped.get_action_mask())


def unique_run():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid4().hex[:8]


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2), encoding="utf-8")


def run_training():
    suite = validate_training()  # Validate every phase before creating run folders.
    torch.set_num_threads(1)
    results = []
    root = Path(OUTPUT_ROOT).expanduser() / EXPERIMENT_NAME / unique_run()
    for name, phases in suite.items():
        set_random_seed(SEED)
        folder = root / name
        for part in ("meta", "tb", "checkpoints", "models", "eval"):
            (folder / part).mkdir(parents=True, exist_ok=True)
        requested = dict(n_steps=N_STEPS, batch_size=BATCH_SIZE, n_epochs=N_EPOCHS,
                         learning_rate=LEARNING_RATE, gamma=GAMMA, ent_coef=ENT_COEF, net_arch=NET_ARCH)
        write_json(folder / "meta" / "run.json", dict(source_commit=SOURCE_COMMIT, notebook_version="1.0",
                   learner_role=LEARNER_ROLE, seed=SEED, device=DEVICE, num_envs=NUM_ENVS,
                   experiment_name=EXPERIMENT_NAME, phases=phases, requested_ppo=requested,
                   resume_model_path=RESUME_MODEL_PATH))
        model = None
        for index, phase in enumerate(phases, 1):
            envs, vec, logger = [], None, None
            try:
                for rank in range(NUM_ENVS):
                    env = make_notebook_env(phase, LEARNER_ROLE, DEVICE)
                    envs.append(env)
                vec = DummyVecEnv([lambda env=env: env for env in envs])
                vec.seed(SEED)
                if model is None:
                    if RESUME_MODEL_PATH:
                        model = MaskablePPO.load(RESUME_MODEL_PATH, env=vec, device=DEVICE)
                        print("Resuming with saved PPO hyperparameters and architecture.")
                    else:
                        model = MaskablePPO("MlpPolicy", vec, device=DEVICE, seed=SEED, verbose=1,
                                            n_steps=N_STEPS, batch_size=BATCH_SIZE, n_epochs=N_EPOCHS,
                                            learning_rate=LEARNING_RATE, gamma=GAMMA, ent_coef=ENT_COEF,
                                            policy_kwargs={"net_arch": NET_ARCH})
                else:
                    model.set_env(vec)
                logger = configure(str(folder / "tb" / f"phase_{index:02}"), ["stdout", "tensorboard"])
                model.set_logger(logger)
                stats = WinStatsCallback(log_every_episodes=10)
                callbacks = [stats]
                if phase["checkpoints_per_run"]:
                    freq = max(phase["timesteps"] // phase["checkpoints_per_run"] // NUM_ENVS, 1)
                    callbacks.append(CheckpointCallback(save_freq=freq, save_path=str(folder / "checkpoints"),
                                                        name_prefix=f"phase_{index:02}"))
                effective = dict(n_steps=model.n_steps, batch_size=model.batch_size, n_epochs=model.n_epochs,
                                 gamma=model.gamma, ent_coef=model.ent_coef, net_arch=model.policy.net_arch,
                                 learning_rate_at_start=float(model.lr_schedule(model._current_progress_remaining)))
                settings = dict(source_commit=SOURCE_COMMIT, learner_role=LEARNER_ROLE, phase=phase,
                                effective_knobs={**DEFAULT_KNOBS, **phase["reward_overrides"]},
                                seed=SEED, device=DEVICE, num_envs=NUM_ENVS, effective_ppo=effective)
                settings_path = folder / "meta" / f"phase_{index:02}_settings.json"
                write_json(settings_path, settings)
                before = model.num_timesteps
                model.learn(total_timesteps=phase["timesteps"], callback=CallbackList(callbacks),
                            reset_num_timesteps=(index == 1 and RESUME_MODEL_PATH is None))
                model_path = folder / "models" / f"phase_{index:02}_final.zip"
                model.save(model_path)
                settings.update(actual_timesteps=model.num_timesteps - before,
                                total_model_timesteps=model.num_timesteps,
                                outcomes=dict(goat_wins=stats.goat_wins, tiger_wins=stats.tiger_wins,
                                              max_timeouts=stats.max_timeouts, repeat_timeouts=stats.repeat_timeouts))
                write_json(settings_path, settings)
                print(f"Saved: {model_path}")
            finally:
                if vec is not None:
                    vec.close()
                else:
                    for env in envs:
                        env.close()
                if logger is not None:
                    logger.close()
        results.append(dict(variation=name, model_path=str(model_path.resolve()),
                            settings_path=str(settings_path.resolve()), settings=settings,
                            eval_dir=str((folder / "eval").resolve()), actual_timesteps=model.num_timesteps))
    return results

<a id="monitor-tensorboard"></a>

## Monitor training with TensorBoard
Run the next cell **before training**. It monitors the configured OUTPUT_ROOT,
including all experiment and phase subfolders. Select runs in the sidebar.

In Colab, the cell uses the built-in TensorBoard magic to display the dashboard through Colab's proxy.

Locally, the cell reuses a responsive server for the same log directory within this kernel.
Otherwise, it starts TensorBoard on an available port and prints the dashboard URL.
**If the embedded dashboard is blank, open the printed URL in your browser.**
Use that URL instead of a saved localhost:6006 or localhost:6007 link, since the port
is chosen automatically. Rerun the TensorBoard code cell after changing OUTPUT_ROOT.

There may be **no dashboards** until training writes its first logs. **Refresh** to see
new results. You can skip the TensorBoard code cell when only evaluating a saved model.
The local server ends when the kernel exits. Colab manages its magic-launched server separately.
No manual kill is normally needed between training runs.

For remote notebooks other than Colab, forward the printed port or use your host's port proxy.
The localhost URL is intended for a locally running kernel.

Alternatively, run tensorboard --logdir artifacts --port 6006 in a terminal,
open http://localhost:6006/, and stop that server with Ctrl+C when finished.


In [ ]:
# Resolve OUTPUT_ROOT to an absolute path so TensorBoard watches the same folder
# on repeated runs. It reads training logs from all experiment/phase subfolders.
# Creating the folder lets this cell run before training has written any logs.
_tb_logdir = str(Path(OUTPUT_ROOT).expanduser().resolve())
Path(_tb_logdir).mkdir(parents=True, exist_ok=True)

# Colab runs the kernel on a remote machine. Its TensorBoard magic supplies
# the proxy needed to display that remote server in your browser.
# A blank iframe does not raise a Python exception, so select by environment
# instead of waiting for the local launcher to fail.

if in_colab:
    print(f"Opening TensorBoard in Colab. Logs: {_tb_logdir}")
    # Equivalent to %load_ext tensorboard and %tensorboard --logdir ... .
    # Using run_line_magic lets us put these commands inside an if block.
    # Quote the configured path so directories containing spaces also work.
    shell.run_line_magic("load_ext", "tensorboard")
    shell.run_line_magic("tensorboard", "--logdir " + shlex.quote(_tb_logdir))
else:
    # Store servers in the notebook's global state, keyed by log directory.
    # Rerunning this cell keeps the dictionary; restarting the kernel clears it.
    # Changing OUTPUT_ROOT selects a different entry without stopping other servers.
    if "_tb_servers" not in globals():
        _tb_servers = {}
    _tb_entry = _tb_servers.get(_tb_logdir)

    # A saved URL does not guarantee that its server still responds.
    # Check it with a short timeout before deciding whether to reuse it.
    _tb_alive = False
    if _tb_entry is not None:
        try:
            with urlopen(_tb_entry["url"], timeout=3) as response:
                _tb_alive = response.status == 200
        except (OSError, ValueError):
            # Connection failures/timeouts or an invalid URL trigger a fresh launch.
            pass

    if not _tb_alive:
        _tb_app = program.TensorBoard()
        # 127.0.0.1 makes the dashboard accessible only on the kernel's computer.
        # Port 0 asks the OS for an available port, avoiding old server conflicts.
        # Always use the printed URL below: the selected port can change.
        _tb_app.configure(argv=[
            "tensorboard", "--logdir", _tb_logdir,
            "--host", "127.0.0.1", "--port", "0",
        ])
        # launch() starts a background server thread and returns its actual URL.
        # Training can run in the next cell; this server ends when the kernel exits.
        _tb_entry = {"app": _tb_app, "url": _tb_app.launch()}
        _tb_servers[_tb_logdir] = _tb_entry

    # Print the watched folder to help diagnose missing runs, plus a browser URL
    # to use if the notebook editor does not display the embedded dashboard.
    _tb_url = _tb_entry["url"]
    print(f"TensorBoard logs: {_tb_logdir}")
    print(f"Open this dashboard URL in your browser: {_tb_url}")

    # A direct iframe src avoids the JavaScript used by the %tensorboard magic.
    # No dashboards may appear until training writes its first logs; then refresh.
    # Remote kernels require port forwarding or the notebook host's port proxy.
    display(IFrame(src=_tb_url, width="100%", height=500))


TensorBoard logs: C:\Users\chris\OneDrive\Documents\GitHub\TNG_Falcon\artifacts
Open this dashboard URL in your browser: http://127.0.0.1:56302/


<a id="run-training"></a>

## Run training
Execute the next code cell for fresh or resumed training. Each variation is independent;
phases continue its model. Paths printed below include final models and matching
settings files. Skip that code cell when evaluating an existing model.

In [ ]:
training_results = run_training()
display(pd.DataFrame([{k: r[k] for k in ("variation", "model_path", "settings_path", "actual_timesteps")} for r in training_results]))

Using cpu device
Logging to artifacts\First_Test\20261018T070425_787030Z_6014a278\baseline\tb\phase_01
----------------------------------------------------
| 1.episode_stats (total episodes)/    |           |
|    1.episode_index                   | 60        |
|    2.goat_win_rate                   | 0         |
|    3.tiger_win_rate                  | 1         |
|    4.max_timeout_rate                | 0         |
|    5.repeat_timeout_rate             | 0         |
| 2.window_stats (per 1000 eps)/       |           |
|    1.episodes_tracked                | 51        |
|    2.goat_win_rate                   | 0         |
|    3.tiger_win_rate                  | 1         |
|    4.max_timeout_rate                | 0         |
|    5.repeat_timeout_rate             | 0         |
|    6.realized_greedy_tiger_frac      | 1         |
|    7.realized_smart_tiger_frac       | 0         |
|    8.realized_random_goat_frac       | 1         |
|    9.realized_model_goat_frac        | 0       

KeyboardInterrupt: 

<a id="save-outputs"></a>

## Save outputs and continue later
Each run lives in `OUTPUT_ROOT/EXPERIMENT_NAME/<unique_run>/<variation>/` with `meta`, `tb`,
`checkpoints`, `models`, and `eval` subfolders. Keep the model **and** its
`phase_*_settings.json` metadata. In Colab, download these files from the Files panel
before ending the session, or set OUTPUT_ROOT to an already-mounted persistent folder.

The TensorBoard cell above Run training monitors the configured `OUTPUT_ROOT`.
Use the same cell to inspect saved logs after training finishes.

To resume, set RESUME_MODEL_PATH in the top configuration cell and select additional
timesteps and matchup settings. Restart and Run All. Saved PPO hyperparameters and
architecture are retained; requested and effective values are recorded in metadata.
This is continued training, not a bit-for-bit replay of the original random state.

This notebook is a portable snapshot, not a live import: repository updates must be
explicitly incorporated into it. Source commit: `1b8ba721253530dad0be529a21628353b3db59df`.


<a id="evaluate-models"></a>

## Evaluate models (independent section)
Evaluation tests how well a model plays. It does **not** train it or change what it learned.
Run the next two code cells after choosing the setting the evaluation settings in the configuration cell at the top of the notebook.

| Setting | What it does |
| --- | --- |
| `EVAL_MODEL_PATH` | `None`: test the final model from each variation trained in this session. A `.zip` path: test that saved model instead. |
| `EVAL_SETTINGS_PATH` | For a saved model, use its `phase_*_settings.json` file to load its learner, reward and game settings. `None` uses the current settings above. |
| `EVAL_OPPONENTS` | A list of opponents to test separately. Goats: `["greedy", "smart"]`; tigers: `["random", "model"]`. `None` uses the model's final training matchup, including mixing. |
| `EVAL_EPISODES` | Number of games **per model, per opponent**. More games give a more useful win rate. |
| `EVAL_SEED` | Starting number for repeatable tests. Each comparison starts from this seed. |
| `EVAL_DETERMINISTIC` | `True`: choose the most likely legal move. `False`: sample a move from the model's choices. |
| `GOAT_MODEL_PATH` | Saved goat model needed when testing a tiger against Model Goat. |

For example, **2 variations × 2 opponents × 100 games = 400 games**.
A variation with several training phases is tested using its **final model**, not once per phase.
Choosing an opponent list replaces any training mix with separate fixed-opponent tests.

To test a saved model, set `EVAL_MODEL_PATH` and preferably `EVAL_SETTINGS_PATH`.
Run configuration, dependency setup, environment, training helpers, then these evaluation
cells. Skip **Run training**. Without saved settings, match the current learner, reward
and game settings to the original run. The environment's game rules must also match.

Results show wins, losses, timeouts, win rate, average reward and average game length.
Each comparison saves CSV/JSON files in its evaluation folder, including the settings used.
A short test checks that everything runs; use more games before judging model strength.

In [ ]:
def evaluation_targets(training_results):
    if EVAL_MODEL_PATH is not None:
        model_path = checked_model_path(EVAL_MODEL_PATH, "EVAL_MODEL_PATH")
        if EVAL_SETTINGS_PATH is not None:
            path = Path(EVAL_SETTINGS_PATH).expanduser()
            if not path.is_file():
                raise ValueError(f"EVAL_SETTINGS_PATH does not exist: {path}")
            settings = json.loads(path.read_text(encoding="utf-8"))
            if not isinstance(settings, dict) or settings.get("learner_role") not in {"goat", "tiger"} or not isinstance(settings.get("phase"), dict):
                raise ValueError("EVAL_SETTINGS_PATH must contain a saved phase settings object.")
            if settings.get("source_commit") != SOURCE_COMMIT:
                print("Note: this model's metadata references a different environment source commit.")
        else:
            settings = dict(learner_role=LEARNER_ROLE, phase=base_phase(), source_commit=SOURCE_COMMIT)
            print("Using global matchup/reward settings; ensure they match the saved model.")
        return [dict(variation="existing_model", model_path=model_path, settings=settings,
                     eval_dir=str(Path(OUTPUT_ROOT).expanduser() / EXPERIMENT_NAME / "eval"))]
    if EVAL_SETTINGS_PATH is not None:
        raise ValueError("EVAL_SETTINGS_PATH requires EVAL_MODEL_PATH.")
    if not training_results:
        raise ValueError("Run training first, or set EVAL_MODEL_PATH to an existing .zip.")
    return copy.deepcopy(training_results)


def run_evaluation(training_results):
    validate_common()
    positive_int(EVAL_EPISODES, "EVAL_EPISODES")
    positive_int(EVAL_SEED, "EVAL_SEED", 0)
    if EVAL_SEED + EVAL_EPISODES > 2**32:
        raise ValueError("EVAL_SEED + EVAL_EPISODES must be <= 2**32.")
    if not isinstance(EVAL_DETERMINISTIC, bool):
        raise ValueError("EVAL_DETERMINISTIC must be True or False.")
    if EVAL_OPPONENTS is not None and (not isinstance(EVAL_OPPONENTS, list) or not EVAL_OPPONENTS or any(not isinstance(v, str) for v in EVAL_OPPONENTS)):
        raise ValueError("EVAL_OPPONENTS must be None or a nonempty list of opponent names.")
    targets = evaluation_targets(training_results)
    jobs = []
    for target in targets:
        checked_model_path(target["model_path"], "evaluation model")
        settings = target["settings"]
        role = settings["learner_role"]
        phase = copy.deepcopy(settings["phase"])
        # Use saved game/reward settings, including defaults, when metadata exists.
        if "effective_knobs" in settings:
            phase["reward_overrides"] = copy.deepcopy(settings["effective_knobs"])
        if GOAT_MODEL_PATH is not None:
            phase["goat_model_path"] = GOAT_MODEL_PATH  # Allows moved/uploaded opponent files.
        opponents = [None] if EVAL_OPPONENTS is None else EVAL_OPPONENTS
        for opponent in opponents:
            selected = copy.deepcopy(phase)
            if opponent is not None:
                selected.update(opponent=opponent, mix_prob=None)
            validate_phase(selected, role)
            jobs.append((target, role, selected))
    rows = []
    run_id = unique_run()
    for target, role, phase in jobs:
        set_random_seed(EVAL_SEED)
        model = MaskablePPO.load(target["model_path"], device=DEVICE)
        env = make_notebook_env(phase, role, DEVICE)
        totals, lengths, counts = [], [], Counter()
        try:
            if model.observation_space != env.observation_space or model.action_space != env.action_space:
                raise ValueError("Evaluation model has incompatible observation/action spaces.")
            for episode in range(EVAL_EPISODES):
                seed = EVAL_SEED + episode
                set_random_seed(seed)
                obs, _ = env.reset(seed=seed)
                done, reward_sum, length = False, 0.0, 0
                while not done:
                    mask = env.action_masks()
                    if not np.any(mask):
                        raise RuntimeError("No legal action available before an episode ended.")
                    action, _ = model.predict(obs, deterministic=EVAL_DETERMINISTIC, action_masks=mask)
                    obs, reward, terminated, truncated, info = env.step(int(np.asarray(action).item()))
                    reward_sum += float(reward)
                    length += 1
                    done = terminated or truncated
                counts[info.get("winner") or "Other"] += 1
                totals.append(reward_sum)
                lengths.append(length)
        finally:
            env.close()
        learner_name, opponent_name = ("Goat", "Tiger") if role == "goat" else ("Tiger", "Goat")
        row = dict(variation=target["variation"], model_path=target["model_path"], learner=role,
                   opponent=phase["opponent"] if phase["mix_prob"] is None else "mixed",
                   mix_prob=phase["mix_prob"], episodes=EVAL_EPISODES,
                   learner_wins=counts[learner_name], opponent_wins=counts[opponent_name],
                   max_timeouts=counts["MaxTimeout"], repeat_timeouts=counts["RepeatTimeout"],
                   other_endings=sum(v for k, v in counts.items() if k not in {"Goat", "Tiger", "MaxTimeout", "RepeatTimeout"}),
                   win_rate=counts[learner_name] / EVAL_EPISODES,
                   mean_reward=float(np.mean(totals)), mean_length=float(np.mean(lengths)),
                   seed=EVAL_SEED, deterministic=EVAL_DETERMINISTIC)
        folder = Path(target["eval_dir"])
        folder.mkdir(parents=True, exist_ok=True)
        stem = folder / f"evaluation_{run_id}_{len(rows):02}"
        settings = dict(phase=phase, effective_knobs={**DEFAULT_KNOBS, **phase["reward_overrides"]},
                        environment_source_commit=SOURCE_COMMIT, model_source_commit=target["settings"].get("source_commit"),
                        device=DEVICE, evaluation_run_id=run_id)
        write_json(stem.with_suffix(".json"), dict(summary=row, settings=settings,
                   episode_rewards=totals, episode_lengths=lengths))
        with stem.with_suffix(".csv").open("w", newline="", encoding="utf-8") as stream:
            csv_row = {**row, "settings_json": json.dumps(settings)}
            writer = csv.DictWriter(stream, fieldnames=list(csv_row))
            writer.writeheader()
            writer.writerow(csv_row)
        rows.append(row)
        print(f"Evaluation saved: {stem}.json and .csv")
    columns = ["variation", "learner", "opponent", "mix_prob", "episodes", "learner_wins", "opponent_wins",
               "max_timeouts", "repeat_timeouts", "other_endings", "win_rate", "mean_reward", "mean_length"]
    return pd.DataFrame(rows)[columns]

In [ ]:
evaluation_results = run_evaluation(globals().get("training_results", []))
display(evaluation_results)

<a id="human-play"></a>

## Play the game yourself

Adapted from the **Human playable version** in the original `x96_TnG.ipynb`.
This keeps the clickable board and two-click movement controls, using the
`TnGEnv` defined in this notebook's environment cell for both sides.

Run the configuration, dependency setup, and environment definition cells first,
then the three code cells below. Training, evaluation, and TensorBoard are optional
for human play; no saved model is required. Dependency setup installs the local
`ipywidgets==8.1.8` pin; the widget setup below enables widget support in Colab.

**VS Code:** version 8.1.9 has a reported blank-widget issue. After installing the
pinned version, restart the notebook kernel, then rerun the configuration, dependency setup,
environment definition, and three human-play code cells. Training and evaluation
do not need to run. Old saved widget output is not an active game; rerun the final
code cell to create a fresh board.
[Compatibility issue](https://github.com/microsoft/vscode-jupyter/issues/17504).

In the final code cell, choose `PLAY_AS = "goat"` or `"tiger"`:

- **Goats:** click an empty position to place a goat. After all 15 have been placed,
  click a goat and then a yellow destination. Choose Greedy or Smart Tigers with
  `PLAY_TIGER_AI`.
- **Tigers:** click a tiger and then a yellow destination, including capture
  landings. The environment plays Random Goats automatically.
- Click your selected piece again to cancel, or another piece of your side to
  change the selection. Blue pieces are goats; red pieces are tigers.
- **New game** resets the board. After changing the play settings, rerun the final
  code cell to replace the old board. A game ends on a win, timeout, or repetition
  limit; the status above the board reports the result.

Play settings are separate from the training role and opponent. Game rules and
rewards use `REWARD_OVERRIDES` from the configuration cell; `PLAY_MAX_TURNS` sets
the human game's turn limit. This section does not train or update a model.


In [ ]:
# Dependencies, imports, and in_colab are defined in Dependency setup.
# Colab needs its widget manager for the clickable board.
if in_colab:
    output.enable_custom_widget_manager()


In [ ]:
# Adapted from HumanBoardWrapper in the original x96_TnG.ipynb; game rules stay in TnGEnv above.
class HumanBoardWrapper:
    def __init__(self, env):
        self.env = env
        # The notebook uses one environment class for both human roles.
        self.is_tiger_env = env.learner_role == "tiger"
        self.selected_piece = None
        self.done = False
        self.closed = False
        self.info = {}
        self.message = ""
        self.state, _ = self.env.reset()

        self.status = widgets.HTML()
        self.buttons = [self._make_button(i) for i in range(BOARD_SIZE)]
        self.restart_button = widgets.Button(description="New game", icon="refresh")
        self.restart_button.on_click(self.reset_game)

        # Preserve the original x96_TnG.ipynb board spacing and numbered square buttons.
        self.spacers = [widgets.Label("", layout=widgets.Layout(width=width))
                        for width in ("120px", "40px")]
        self.rows = [
            widgets.HBox([self.spacers[0], self.buttons[0]]),
            widgets.HBox(self.buttons[1:7]),
            widgets.HBox(self.buttons[7:13]),
            widgets.HBox(self.buttons[13:19]),
            widgets.HBox([self.spacers[1], *self.buttons[19:23]]),
        ]
        self.board_ui = widgets.VBox([self.status, *self.rows, self.restart_button])
        self.update_board()

    def _make_button(self, idx):
        button = widgets.Button(description=str(idx),
                                layout=widgets.Layout(width="40px", height="40px"))
        button.on_click(lambda _button, position=idx: self.click(position))
        return button

    def legal_destinations(self, piece_idx):
        """Map clickable destinations to legal actions from the selected piece."""
        mask = self.env.get_action_mask()
        if self.is_tiger_env:
            # Use the environment's actual capture landing and direction rules.
            # The original x96_TnG.ipynb _resolve_tiger_action method is not used here.
            return {
                dest: self.env.encode_action(start, direction)
                for start, dest, _capture, direction in self.env._tiger_moves(include_dir=True)
                if start == piece_idx and mask[self.env.encode_action(start, direction)]
            }
        return {
            dest: self.env.encode_action(piece_idx, direction)
            for direction, dest in self.env.move_map.get(piece_idx, {}).items()
            if dest is not None and mask[self.env.encode_action(piece_idx, direction)]
        }

    def click(self, idx):
        """Place a goat, select a piece, or move to a highlighted destination."""
        if self.done or self.closed:
            return
        self.message = ""
        if not self.is_tiger_env and self.env.phase == 0:
            action = self.env.encode_action(idx, 0)
            if self.env.get_action_mask()[action]:
                self._apply_action(action)
            else:
                self.message = "Choose an empty position to place a goat."
        else:
            own_piece = 2 if self.is_tiger_env else 1
            if idx == self.selected_piece:
                self.selected_piece = None
            elif self.env.board[idx] == own_piece:
                self.selected_piece = idx
                if not self.legal_destinations(idx):
                    self.message = "That piece has no legal moves; select another."
            elif self.selected_piece is not None:
                action = self.legal_destinations(self.selected_piece).get(idx)
                if action is None:
                    self.message = "Choose a yellow destination or select another piece."
                else:
                    self._apply_action(action)
            else:
                self.message = "Select one of your pieces first."
        self.update_board()

    def _apply_action(self, action):
        # One environment step includes the human move and the opponent's reply.
        self.state, _reward, terminated, truncated, self.info = self.env.step(action)
        self.done = bool(terminated or truncated)
        self.selected_piece = None

    def update_board(self):
        destinations = self.legal_destinations(self.selected_piece) if self.selected_piece is not None else {}
        for idx, button in enumerate(self.buttons):
            piece = int(self.env.board[idx])
            button.description = str(idx)
            button.tooltip = f"{COORD_LABELS[idx]} (position {idx})"
            button.style.button_color = {0: None, 1: "lightblue", 2: "lightcoral"}[piece]
            if idx in destinations:
                button.style.button_color = "yellow"
            button.disabled = self.done or self.closed

        role = "Tigers" if self.is_tiger_env else "Goats"
        opponent = "Random Goats" if self.is_tiger_env else f"{self.env.tiger_ai.title()} Tigers"
        phase = "Place goats" if self.env.phase == 0 else "Move goats"
        status = (
            f"<b>You: {role} | Opponent: {escape(opponent)}</b><br>"
            f"{phase} | Goats placed: {self.env.goats_placed}/{TOTAL_GOATS_TO_PLACE} | "
            f"Goats eaten: {self.env.eaten}/{GOATS_EATEN_FOR_TIGER_WIN} | "
            f"Turn: {self.env.turns}/{self.env.max_turns}"
        )
        if self.done:
            winner = self.info.get("winner")
            result = f"{winner} wins" if winner in {"Goat", "Tiger"} else "Game ended"
            reason = self.info.get("reason") or winner or "episode ended"
            status += f"<br><b>{escape(result)}: {escape(str(reason))}. Click New game to play again.</b>"
        elif self.message:
            status += "<br>" + escape(self.message)
        elif not self.is_tiger_env and self.env.phase == 0:
            status += "<br>Click an empty position to place a goat."
        else:
            status += "<br>Select your piece, then click a yellow destination."
        self.status.value = status + (
            '<br><b>Legend:</b> '
            '<span style="background-color: lightblue; color: black; padding: 2px 6px;">Goat</span> | '
            '<span style="background-color: lightcoral; color: black; padding: 2px 6px;">Tiger</span> | '
            '<span style="background-color: yellow; color: black; padding: 2px 6px;">Legal move</span> | '
            'Uncolored = empty'
        )

    def reset_game(self, _button=None):
        if self.closed:
            return
        self.state, _ = self.env.reset()
        self.done = False
        self.selected_piece = None
        self.info = {}
        self.message = ""
        self.update_board()

    def show(self):
        display(self.board_ui)

    def close(self):
        # Rerunning the launch cell retires the old game and its widget callbacks.
        self.closed = True
        self.env.close()
        for widget in [self.board_ui, self.status, self.restart_button, *self.rows, *self.spacers, *self.buttons]:
            widget.close()


In [ ]:
# HUMAN PLAY SETTINGS: independent of the training role/opponent above.
PLAY_AS = "goat"              # "goat" or "tiger"
PLAY_TIGER_AI = "smart"      # "greedy" or "smart"; used when you play goats
PLAY_MAX_TURNS = 100          # Positive turn limit for this human game

if PLAY_AS not in {"goat", "tiger"}:
    raise ValueError('PLAY_AS must be "goat" or "tiger".')
if PLAY_TIGER_AI not in {"greedy", "smart"}:
    raise ValueError('PLAY_TIGER_AI must be "greedy" or "smart".')
if isinstance(PLAY_MAX_TURNS, bool) or not isinstance(PLAY_MAX_TURNS, int) or PLAY_MAX_TURNS <= 0:
    raise ValueError("PLAY_MAX_TURNS must be a positive integer.")

# Use the environment and reward settings already defined in this notebook.
# Human Tigers face Random Goats, matching the original x96_TnG.ipynb playable wrapper.
play_env = TnGEnv(learner_role=PLAY_AS, tiger_ai=PLAY_TIGER_AI,
                  goat_opponent_ai="random", reward_weights=dict(REWARD_OVERRIDES),
                  max_turns=PLAY_MAX_TURNS)
previous_game = globals().get("human_game")
if previous_game is not None:
    previous_game.close()
human_game = HumanBoardWrapper(play_env)
human_game.show()
